# 07. Evidence Analysis Foundry Component Build

This notebook builds the real Microsoft Foundry pieces of the Carrot Tuning system one component at a time. It follows the useful code-first pattern from the FrontierWeekHack claims example: Python constructs a narrowly scoped Foundry component, checks what Azure returned, and leaves a readable history.

The similarity ends at the safety boundary. The claims example creates tool-using agents that recommend insurance actions. This workbench will create one **no-tools evidence analyst** only after deterministic local gates approve a redacted packet. It will not mount the repository, call Kaggriculture, edit source, create a Knowledge index, or independently recommend a parameter value.

## Development envelope and stop points

Safe notebook work may inspect project metadata, construct configuration records, create SDK request objects, and render those facts. Cloud mutations remain separate cells with an exact target and a human approval immediately before them. A successful model deployment does not authorize agent creation; a successful agent creation does not authorize a model invocation.

- [ ] 1. Record and validate the proposed model-deployment configuration.
- [x] 2. Read-only Foundry discovery.
- [x] 3. Build and review the exact model-deployment request.
- [x] 4. Create and verify the model deployment.
- [x] 5. Build and review the no-tools prompt-agent definition from source-controlled instructions.
- [x] 6. Create and verify the prompt agent.
- [ ] 7. After approval: invoke one redacted fixed fixture and inspect its trace/output gates.
- [ ] Future hardening: retrofit notebooks 01–06 with this same rerunnable resource pattern.

Each unchecked item is an explicit review boundary, not permission to run the later cells.

## Rerun and idempotence policy

Every cell is executable in dependency order and safe to rerun within its declared boundary. Pure records and renderers recreate the same values; read-only checks may observe the same service without mutation; the deployment apply reads first, reuses an exact match, and refuses drift; prompt-agent creation lists first and creates only when the named agent is absent; and the model invocation cell reuses its recorded result instead of sending a second message. Historical receipts supplement executable guarded actions. If live state changes, the notebook preserves the original pre-apply snapshot instead of silently treating the new state as the old precondition.

## Handoff: what we are building, and what happens next

This notebook is the source-visible build history for the first remote component of the Carrot Tuning system. It is designed to let the owner—and a successor who has not seen the prior conversation—understand the target architecture, inspect each proposed cloud change before it happens, and see exactly which decisions still belong to a human.

### The component in one sentence

We are building one stateless, **no-tools Foundry prompt agent** that receives a redacted, deterministic-local-evaluator-approved evidence brief and returns a constrained explanation for human review. It is an evidence analyst, not a farmer, game runner, optimizer, policy editor, or experiment approver.

### Authority map

- The local Kaggriculture evaluator remains the truth owner for target binding, simulation, matched outcomes, clean audits, and immutable evidence packets.
- The local `analyst/` package remains the truth owner for packet eligibility, redacted-brief construction, and deterministic response gating.
- Foundry receives only the approved redacted brief. Its first prompt agent may explain packet-backed facts, cite them, state limits, and leave `pending_human_review`; it may not use tools or reach the repository.
- A deterministic local gate, not the model, decides whether a response is structurally eligible. The human alone approves a new experiment envelope or a source-policy change.

### What already exists

- Notebook 06 and the local `analyst/` package already establish the immutable request/response shape, fail-closed eligibility gate, redacted brief, constrained instruction policy, and deterministic output gate. Their fast tests are simulator-free.
- The Foundry account `foundry-trainer-resource` and project `foundry-trainer` are in East US 2, inside `rg-foundry-trainer-eastus2`. Redacted traces have a 30-day policy and the resource group has a $5 monthly alerting budget.
- The model deployment and no-tools `evidence-analyst` prompt agent were deleted for a clean rebuild. No tool, connector, Knowledge index, model call, or model-generated trace is assumed to exist until the rerun reads and recreates it.
- Section 1 records the desired state for the rebuild: `evidence-analyst-luna-v1`, `gpt-5.6-luna` version `2026-07-09`, `DataZoneStandard`, capacity units `6` (6,000 TPM), default content filter, no spillover, and `NoAutoUpgrade`.

### How this notebook will proceed

This workbench deliberately separates five kinds of work: desired-state record, read-only discovery, exact request construction, side-effecting apply, and Azure readback verification. A success in one category never authorizes the next. The notebook is the sole Azure action authority: if a cloud cell is missing or broken, repair that cell rather than falling back to MCP, CLI, a standalone SDK script, or the portal. The near-term sequence is:

- Record and locally validate the desired model deployment.
- Inspect the SDK, project, deployments, and prompt agents read-only.
- Construct and review the exact deployment request.
- Only after explicit approval, apply that request and read it back.
- Construct and review the no-tools prompt-agent definition from the versioned local instructions.
- Only after separate approval, create and read back that prompt agent.
- Only after another approval, invoke one fixed redacted fixture and inspect its deterministic gates and redacted trace.

Each future automated decision must retain `review_status: pending_human_review` and a visible unchecked `- [ ]` task until the owner records a disposition.

### FrontierWeekHack sample: useful pattern, not authority

The local `FrontierWeekHack/claims` repository is a read-only example of course-style Foundry client setup, create/list/readback patterns, and agent lifecycle code. It can help us make the Python workbench legible. It does **not** authorize us to copy its tool-using claims agents, action recommendations, connections, credentials, cleanup behavior, or deployment commands. We do not modify or run that sample.

### Success criteria for this notebook

The notebook will be successful when it contains a readable, reviewed, source-visible path from the local evidence boundary through a minimal Foundry prompt agent and a fixed-fixture evaluation/tracing demonstration—without silently expanding the agent's authority. It is not successful merely because an Azure resource exists or a model produces fluent text.

### Immediate successor task

Study the claims sample read-only, report the small set of SDK patterns that safely map to this project, then keep component 2 as a **non-mutating** notebook section. Its cells may run in order and must use read-only guards; they must not alter the sample repository or install unreviewed dependencies.

For full continuity, read `docs/next-agent-foundry-notebook-handoff.md`, `docs/foundry-transition-handoff.md`, `docs/foundry-provisioning-decision-record.md`, `README.md`, `docs/current-state-and-next-slice.md`, and the Foundry section of `QUESTION_LEDGER.md` before making a substantive change.

## 0. Tracing bootstrap prerequisite

The Microsoft sample sets tracing environment variables before importing Azure SDK clients, instruments `AIProjectClient`, and configures the Azure Monitor exporter with the Application Insights connection string. This section mirrors that order.

⚠️ **Assumption:** Full GenAI tracing captures message content and may expose prompts/responses in telemetry. This project sends a redacted fixture only; any broader capture is a separate human decision.

❌ If `azure-monitor-opentelemetry` is not installed, this cell records the exact prerequisite and stops. It does not install packages automatically.

In [1]:
import os
from importlib.util import find_spec

# These flags must be set before importing azure.ai.projects in later cells.
os.environ['AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING'] = 'true'
os.environ['OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT'] = 'true'

try:
    tracing_exporter_spec = find_spec('azure.monitor.opentelemetry')
except ModuleNotFoundError:
    tracing_exporter_spec = None
if tracing_exporter_spec is None:
    TRACING_BOOTSTRAP_RESULT = {
        'status': 'blocked',
        'error': '❌ azure-monitor-opentelemetry is not installed; install the reviewed sample dependency before rerunning this bootstrap',
        'env_flags_set': True,
    }
    raise RuntimeError(TRACING_BOOTSTRAP_RESULT['error'])

from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.identity import DefaultAzureCredential
import json
import urllib.request

# The exporter receives the connection string in memory; it is never displayed or stored.
credential = DefaultAzureCredential()
try:
    appinsights_id = '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights'
    token = credential.get_token('https://management.azure.com/.default').token
    request = urllib.request.Request(
        f'https://management.azure.com{appinsights_id}?api-version=2020-02-02',
        headers={'Authorization': f'Bearer {token}'},
    )
    with urllib.request.urlopen(request) as response:
        appinsights_body = json.loads(response.read().decode('utf-8'))
    properties = appinsights_body.get('properties', {})
    connection_string = properties.get('ConnectionString') or properties.get('connectionString')
    if not connection_string:
        raise RuntimeError('❌ Application Insights connection string was not available for the tracing exporter')
    AIProjectInstrumentor().instrument()
    from azure.monitor.opentelemetry import configure_azure_monitor
    configure_azure_monitor(connection_string=connection_string, enable_live_metrics=True)
    TRACING_BOOTSTRAP_RESULT = {'status': 'available', 'env_flags_set': True, 'instrumentor_configured': True, 'exporter_configured': True}
finally:
    credential.close()
TRACING_BOOTSTRAP_RESULT


{'status': 'available', 'env_flags_set': True, 'instrumentor_configured': True, 'exporter_configured': True}

### 0.1 Pretty-print tracing bootstrap

The card shows whether instrumentation was actually configured. It never displays the connection string.

In [2]:
from IPython.display import Markdown, display

result = globals().get('TRACING_BOOTSTRAP_RESULT', {'status': 'blocked', 'error': '❌ tracing bootstrap did not produce a result'})
marker = '❌' if result.get('status') == 'blocked' else '✅'
display(Markdown(f"""### {marker} Tracing bootstrap

| Fact | Value |
|:---|:---|
| Status | **{result.get('status', 'unknown')}** |
| Environment flags set | `{result.get('env_flags_set', False)}` |
| Instrumentor configured | `{result.get('instrumentor_configured', False)}` |
| Exporter configured | `{result.get('exporter_configured', False)}` |
| Error | `{result.get('error', 'None')}` |
"""))


### ✅ Tracing bootstrap

| Fact | Value |
|:---|:---|
| Status | **available** |
| Environment flags set | `True` |
| Instrumentor configured | `True` |
| Exporter configured | `True` |
| Error | `None` |


## 1. Record the proposed rebuild deployment

This is a pure Python record of the rebuild deployment—not a call to Azure. `capacity_units=6` is the Azure deployment value for 6,000 TPM. Data Zone Standard keeps inference processing in Microsoft's US data zone; it is pay-per-token, not provisioned throughput. `NoAutoUpgrade` makes any later model-version change a deliberate review rather than an invisible change to the evidence chain.

The later apply component must consume this reviewed record rather than retyping model facts into a portal or a shell command.

In [3]:
from dataclasses import dataclass
import re


@dataclass(frozen=True)
class EvidenceAnalystDeploymentConfig:
    """A reviewed desired state; this class performs no cloud operation."""

    subscription_id: str
    resource_group: str
    account_name: str
    project_name: str
    deployment_name: str
    model_name: str
    model_version: str
    model_format: str
    sku_name: str
    capacity_units: int
    content_filter_policy: str
    version_upgrade_policy: str
    spillover_deployment_name: None
    review_status: str


def validate_deployment_config(config: EvidenceAnalystDeploymentConfig) -> EvidenceAnalystDeploymentConfig:
    """Refuse drift from the reviewed, low-cost first-deployment envelope."""
    # Naming is checked before any later code could turn this record into a request.
    if not re.fullmatch(r'[\w.-]{2,64}', config.deployment_name):
        raise ValueError('deployment name must use 2-64 letters, digits, dots, hyphens, or underscores')
    # Pin the model identity so a version change is visible and reviewable.
    if (config.model_name, config.model_version, config.model_format) != (
        'gpt-5.6-luna', '2026-07-09', 'OpenAI',
    ):
        raise ValueError('model identity differs from the reviewed analyst candidate')
    # Six Data Zone units provide enough TPM for a 5,000-token response plus input and safety margin.
    if config.sku_name != 'DataZoneStandard' or config.capacity_units != 6:
        raise ValueError('the rebuild deployment requires six US Data Zone capacity units (6,000 TPM)')
    if config.content_filter_policy != 'Microsoft.DefaultV2':
        raise ValueError('the reviewed default content-filter policy is required')
    # No spillover or silent version change is allowed in the first slice.
    if config.version_upgrade_policy != 'NoAutoUpgrade' or config.spillover_deployment_name is not None:
        raise ValueError('version changes and overflow routing remain explicit later decisions')
    if config.review_status != 'pending_human_review':
        raise ValueError('the deployment stays blocked until explicit human approval')
    # Returning the same frozen value preserves the validated desired state.
    return config


# This is a local proposal, not a portal readback or an Azure request.
EVIDENCE_ANALYST_DEPLOYMENT_CONFIG = EvidenceAnalystDeploymentConfig(
    subscription_id='d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930',
    resource_group='rg-foundry-trainer-eastus2',
    account_name='foundry-trainer-resource',
    project_name='foundry-trainer',
    deployment_name='evidence-analyst-luna-v1',
    model_name='gpt-5.6-luna',
    model_version='2026-07-09',
    model_format='OpenAI',
    sku_name='DataZoneStandard',
    capacity_units=6,
    content_filter_policy='Microsoft.DefaultV2',
    version_upgrade_policy='NoAutoUpgrade',
    spillover_deployment_name=None,
    review_status='pending_human_review',
)
# Validation is intentionally explicit before another cell consumes the record.
VALIDATED_DEPLOYMENT_CONFIG = validate_deployment_config(EVIDENCE_ANALYST_DEPLOYMENT_CONFIG)


In [4]:
def render_deployment_configuration(config: EvidenceAnalystDeploymentConfig) -> None:
    """Presentation only: format an already-validated local configuration."""
    from IPython.display import Markdown, display

    # This renderer reads facts; it does not choose values or call Azure.
    display(Markdown(f"""### Proposed first model deployment

| Setting | Recorded value |
|:---|:---|
| Account / project | `{config.account_name}` / `{config.project_name}` |
| Resource group | `{config.resource_group}` |
| Deployment name | `{config.deployment_name}` |
| Model | `{config.model_name}` `{config.model_version}` |
| SKU / capacity | `{config.sku_name}` / `{config.capacity_units}` units = **6,000 TPM** |
| Content filter / upgrades | `{config.content_filter_policy}` / `{config.version_upgrade_policy}` |
| Spillover | none |
| Azure action | **not performed** |
| Review state | **{config.review_status}** |

- [ ] Review this deployment configuration: approve, revise, or reject.

> This output describes an in-memory configuration only. It does not send an Azure request.
"""))


# The display is separate so it can be hidden or cleared without changing the record.
render_deployment_configuration(VALIDATED_DEPLOYMENT_CONFIG)


### Proposed first model deployment

| Setting | Recorded value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Resource group | `rg-foundry-trainer-eastus2` |
| Deployment name | `evidence-analyst-luna-v1` |
| Model | `gpt-5.6-luna` `2026-07-09` |
| SKU / capacity | `DataZoneStandard` / `6` units = **6,000 TPM** |
| Content filter / upgrades | `Microsoft.DefaultV2` / `NoAutoUpgrade` |
| Spillover | none |
| Azure action | **not performed** |
| Review state | **pending_human_review** |

- [ ] Review this deployment configuration: approve, revise, or reject.

> This output describes an in-memory configuration only. It does not send an Azure request.


## Component 2 context

Component 2 is now complete for the approved read-only scope. On the clean rebuild run, the notebook must record the local SDK prerequisite result, the exact request boundary, and the actual Foundry response: the configured project should be reachable with zero model deployments and zero prompt agents before apply.

The pure request is constructed, validated, and presented from that fresh preflight. The rerun then applies and verifies the deployment, defines and creates the no-tools prompt agent, and reaches step 7 only after those readbacks succeed.

### Claims-sample reference (context only)

The FrontierWeekHack claims sample is useful as a teaching example for three small patterns we may later adapt: keep Foundry client and project wiring explicit; keep create, list, and readback/verification as visibly separate operations; and keep the important configuration and lifecycle decisions in source-controlled Python rather than hiding them in a portal. These are patterns for making our work inspectable, not instructions to copy the sample's authority.

The following claims-system features do **not** transfer to this project: tool use, claims-action authority, insurance recommendations, repository or data-file access, connections, credentials or `.env` handling, cleanup commands, agent creation, deployment, and invocation. The first Carrot Tuning component remains a no-tools evidence analyst that receives only a redacted, locally validated evidence brief.

The discovery framework is now implemented as separate plan, prerequisite, request, execution, and presentation cells. The next agent must preserve the read-only result and review task; it must not treat zero resources as permission to construct or apply a deployment.

## 2. Read-only Foundry discovery

### 2.1 Define the discovery plan

This first implementation slice defines a **plan**, not a cloud reader. It turns the reviewed deployment target into an explicit list of facts that a later approved discovery cell could inspect. Keeping this plan pure lets us review the target and the operation boundary without installing packages, creating credentials, contacting Azure, or guessing what a successful response will look like.

The plan's input is the validated desired-state configuration from section 1. Its output is a frozen, source-visible description of the target endpoint and the three intended read-only observations: project identity/reachability, existing model deployments, and existing prompt agents. It deliberately excludes package installation, credential construction, create/update/delete operations, model invocation, and durable result storage.

The next review point is the plan itself. Only after it is accepted may we decide separately whether to inspect local package metadata, change the environment, or run one authenticated read through a notebook cell.

In [5]:
from dataclasses import dataclass


# Frozen fields make the reviewable plan stable after it is built.
@dataclass(frozen=True)
class ReadOnlyDiscoveryPlan:
    """Pure description of a future read-only Foundry inspection."""

    account_name: str
    project_name: str
    project_endpoint: str
    observations: tuple[str, ...]
    allows_environment_change: bool
    allows_cloud_mutation: bool
    allows_model_invocation: bool
    stores_cloud_output: bool


def build_read_only_discovery_plan(
    config: EvidenceAnalystDeploymentConfig,
) -> ReadOnlyDiscoveryPlan:
    """Build a reviewable discovery plan without importing or calling an SDK."""
    # Reuse the section-1 validator instead of inventing a second target contract.
    validate_deployment_config(config)
    # This URL is a described target only; constructing it performs no network work.
    return ReadOnlyDiscoveryPlan(
        account_name=config.account_name,
        project_name=config.project_name,
        project_endpoint=(
            f"https://{config.account_name}.services.ai.azure.com/api/projects/"
            f"{config.project_name}"
        ),
        observations=(
            "project identity and read reachability",
            "existing model deployments",
            "existing prompt agents",
        ),
        allows_environment_change=False,
        allows_cloud_mutation=False,
        allows_model_invocation=False,
        stores_cloud_output=False,
    )


# The plan is pure and safe to recreate whenever the notebook runs.
DISCOVERY_PLAN = build_read_only_discovery_plan(VALIDATED_DEPLOYMENT_CONFIG)


### 2.2 Presentation-only discovery-plan view

The next cell is presentation-only. If run, it will format the pure plan already built above. It does not inspect packages, authenticate, contact Azure, or invent a discovery result. The displayed checklist is a review of the plan's permissions, not evidence that the future read has happened.

In [6]:
def render_read_only_discovery_plan(plan: ReadOnlyDiscoveryPlan) -> None:
    """Presentation only: summarize a pure discovery plan for review."""
    from IPython.display import Markdown, display

    # The renderer reports declared permissions, not observed Azure state.
    observations = "\n".join(f"- `{item}`" for item in plan.observations)
    display(Markdown(f"""### Read-only discovery plan

| Planned target | Value |
|:---|:---|
| Account / project | `{plan.account_name}` / `{plan.project_name}` |
| Endpoint described | `{plan.project_endpoint}` |
| Future observations |
{observations} |
| Environment change | **not allowed** |
| Cloud mutation | **not allowed** |
| Model invocation | **not allowed** |
| Cloud output storage | **not allowed** |

- [ ] Review this discovery plan before authorizing any local prerequisite or authenticated read.

> This is a description of an in-memory plan. It is not project inventory or cloud evidence.
"""))


# Clearing or hiding this cell does not change DISCOVERY_PLAN.
render_read_only_discovery_plan(DISCOVERY_PLAN)


### Read-only discovery plan

| Planned target | Value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Endpoint described | `https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer` |
| Future observations |
- `project identity and read reachability`
- `existing model deployments`
- `existing prompt agents` |
| Environment change | **not allowed** |
| Cloud mutation | **not allowed** |
| Model invocation | **not allowed** |
| Cloud output storage | **not allowed** |

- [ ] Review this discovery plan before authorizing any local prerequisite or authenticated read.

> This is a description of an in-memory plan. It is not project inventory or cloud evidence.


### 2.3 Define the local SDK prerequisite check

Before a future authenticated read can be attempted, we need to know whether this Python environment already has the reviewed client libraries. This cell defines a **local-only** check of distribution metadata and import-spec availability. It does not import an Azure client, construct a credential, access the network, install anything, or alter the environment.

A missing package is reported as a prerequisite for a later decision. It is not an error to repair automatically, and a passing check would not authorize the cloud reader.

**Competition note:** the official rules require a clear-purpose AI agent, a written description, examples/lessons, and a short demonstration video. They do not require Python, `azure-identity`, `DefaultAzureCredential`, or direct SDK authentication. This check supports an optional code-first inspection path; it is not required merely to submit or demonstrate the Foundry agent. See `references/foundry-competition/README.md`.

In [7]:
from importlib import metadata, util


def inspect_local_sdk_prerequisites() -> dict[str, dict[str, object]]:
    """Report local Foundry SDK presence without importing or installing it."""
    required_packages = (
        ("azure-ai-projects", "azure.ai.projects"),
        ("azure-identity", "azure.identity"),
    )
    results: dict[str, dict[str, object]] = {}

    for distribution_name, module_name in required_packages:
        # Distribution metadata tells us what is installed; it makes no network request.
        try:
            distribution_version = metadata.version(distribution_name)
        except metadata.PackageNotFoundError:
            distribution_version = None

        # Finding a module spec avoids importing the Azure client or running its startup code.
        try:
            module_available = util.find_spec(module_name) is not None
        except ModuleNotFoundError:
            module_available = False

        results[distribution_name] = {
            "module": module_name,
            "distribution_version": distribution_version,
            "module_available": module_available,
        }

    return results


# This approved local-only call reads package metadata but performs no installation or network access.
SDK_PREREQUISITES = inspect_local_sdk_prerequisites()


### 2.4 Presentation-only prerequisite view

This presentation cell formats the actual package metadata produced by the preceding local check. It does not report a cloud connection, authentication success, deployment, or agent.

In [8]:
def render_local_sdk_prerequisites(prerequisites: dict[str, dict[str, object]]) -> None:
    """Presentation only: format an actual local prerequisite result."""
    from IPython.display import Markdown, display

    rows = []
    for package_name, details in prerequisites.items():
        version = details["distribution_version"] or "not installed"
        module_state = "available" if details["module_available"] else "not available"
        rows.append(f"| `{package_name}` | `{version}` | **{module_state}** |")
    table_rows = "\n".join(rows)

    # This renderer reports local facts only; it never recommends installation.
    display(Markdown(f"""### Local SDK prerequisite result

| Distribution | Version | Module |
|:---|:---|:---|
{table_rows}

- [ ] Review the local prerequisite result before considering any environment change or cloud read.

> This is local package metadata, not Foundry project evidence.
"""))


# This approved renderer formats the local result; it does not infer cloud state.
render_local_sdk_prerequisites(SDK_PREREQUISITES)


### Local SDK prerequisite result

| Distribution | Version | Module |
|:---|:---|:---|
| `azure-ai-projects` | `2.7.0` | **available** |
| `azure-identity` | `1.25.3` | **available** |

- [ ] Review the local prerequisite result before considering any environment change or cloud read.

> This is local package metadata, not Foundry project evidence.


### 2.5 Define the authenticated read-only request

The local prerequisite check tells us that the client libraries are present, but it does not prove that this notebook should contact Foundry. This next pure slice turns the reviewed discovery plan into an explicit request record. It names the exact project target and the only observations permitted: project reachability/identity, existing model deployments, and existing prompt agents.

The request record is still not a client, credential, or network call. It stays `pending_human_review` until the owner approves this exact authenticated read. A successful read would not authorize deployment, agent creation, invocation, or any other mutation.

In [9]:
@dataclass(frozen=True)
class ReadOnlyCloudDiscoveryRequest:
    """Exact read boundary; constructing this record performs no cloud work."""

    account_name: str
    project_name: str
    project_endpoint: str
    operations: tuple[str, ...]
    prohibited_operations: tuple[str, ...]
    review_status: str


def build_read_only_cloud_discovery_request(
    plan: ReadOnlyDiscoveryPlan,
) -> ReadOnlyCloudDiscoveryRequest:
    """Convert the pure plan into a reviewable, non-mutating request record."""
    # The plan is the only source of target and operation facts.
    if plan.allows_environment_change or plan.allows_cloud_mutation or plan.allows_model_invocation:
        raise ValueError('discovery request cannot inherit a mutating permission')

    # These are observations, not commands to create, update, delete, or invoke.
    return ReadOnlyCloudDiscoveryRequest(
        account_name=plan.account_name,
        project_name=plan.project_name,
        project_endpoint=plan.project_endpoint,
        operations=(
            "read project identity/reachability",
            "list existing model deployments",
            "list existing prompt agents",
        ),
        prohibited_operations=(
            "install or change packages",
            "create, update, or delete Azure resources",
            "invoke a model or agent",
        ),
        review_status="pending_human_review",
    )


READ_ONLY_CLOUD_DISCOVERY_REQUEST = build_read_only_cloud_discovery_request(DISCOVERY_PLAN)


### 2.6 Presentation-only request review

This card makes the proposed authenticated read concrete enough to approve or reject. It reports the target and the allowed/prohibited operations from the request record. It does not claim that the project was reached or that any resource exists.

In [10]:
def render_read_only_cloud_discovery_request(
    request: ReadOnlyCloudDiscoveryRequest,
) -> None:
    """Presentation only: format the proposed read boundary."""
    from IPython.display import Markdown, display

    allowed = "\n".join(f"- {operation}" for operation in request.operations)
    prohibited = "\n".join(f"- {operation}" for operation in request.prohibited_operations)
    # This output describes authorization scope; it is not a cloud response.
    display(Markdown(f"""### Proposed authenticated read-only inspection

| Target | Value |
|:---|:---|
| Account / project | `{request.account_name}` / `{request.project_name}` |
| Endpoint | `{request.project_endpoint}` |
| Review state | **{request.review_status}** |

**Allowed observations**
{allowed}

**Prohibited operations**
{prohibited}

- [ ] Approve or revise this exact authenticated read before any credential or client is constructed.

> No Azure request has been made by this cell.
"""))


render_read_only_cloud_discovery_request(READ_ONLY_CLOUD_DISCOVERY_REQUEST)


### Proposed authenticated read-only inspection

| Target | Value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Endpoint | `https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer` |
| Review state | **pending_human_review** |

**Allowed observations**
- read project identity/reachability
- list existing model deployments
- list existing prompt agents

**Prohibited operations**
- install or change packages
- create, update, or delete Azure resources
- invoke a model or agent

- [ ] Approve or revise this exact authenticated read before any credential or client is constructed.

> No Azure request has been made by this cell.


### 2.7 Execute the approved read-only inspection

The owner has approved exactly the request shown above. This cell is the first one in component 2 that may contact Foundry. It uses the already-authenticated local Entra/Azure CLI credential and performs only the three approved observations. It does not install packages, write Azure resources, invoke a model, or create an agent.

The result is summarized into counts and names rather than retaining raw SDK objects. Those facts establish what the project returned at this time; they do not authorize the next component.

In [11]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def execute_approved_read_only_discovery(
    request: ReadOnlyCloudDiscoveryRequest,
) -> dict[str, object]:
    """Perform only the approved project/deployment/agent read."""
    if request.review_status != "pending_human_review":
        raise ValueError("the request must retain its explicit review state")

    # This is the approved external boundary: use the existing local Entra login.
    credential = DefaultAzureCredential()
    client = AIProjectClient(endpoint=request.project_endpoint, credential=credential)
    try:
        # These are list/read operations only; no create, update, delete, or invoke call appears here.
        deployments = tuple(client.deployments.list())
        prompt_agents = tuple(client.agents.list())
    finally:
        client.close()
        credential.close()

    # Keep the durable notebook result compact and avoid storing raw SDK objects.
    return {
        "target": {
            "account_name": request.account_name,
            "project_name": request.project_name,
            "endpoint": request.project_endpoint,
        },
        "deployment_names": tuple(getattr(item, "name", "<unnamed>") for item in deployments),
        "prompt_agent_names": tuple(getattr(item, "name", "<unnamed>") for item in prompt_agents),
        "deployment_count": len(deployments),
        "prompt_agent_count": len(prompt_agents),
        "operations_performed": request.operations,
        "mutations_performed": False,
        "model_invocations_performed": False,
    }


FOUNDATION_READ_RESULT = execute_approved_read_only_discovery(READ_ONLY_CLOUD_DISCOVERY_REQUEST)


### 2.8 Descriptive read-result view

This final component-2 renderer formats the actual redacted result returned by the approved read. It separates observed project state from the permissions that remain closed. A zero count means the list was empty at this read time; it does not mean a deployment or agent was created.

In [12]:
def render_read_only_discovery_result(result: dict[str, object]) -> None:
    """Presentation only: format the approved read's redacted facts."""
    from IPython.display import Markdown, display

    deployment_names = ", ".join(result["deployment_names"]) or "none"
    prompt_agent_names = ", ".join(result["prompt_agent_names"]) or "none"
    # The renderer reports observations and explicitly repeats what did not occur.
    display(Markdown(f"""### Approved read-only Foundry result

| Observed field | Result |
|:---|:---|
| Account / project | `{result['target']['account_name']}` / `{result['target']['project_name']}` |
| Model deployments | **{result['deployment_count']}** ({deployment_names}) |
| Prompt agents | **{result['prompt_agent_count']}** ({prompt_agent_names}) |
| Azure mutations | **none** |
| Model invocations | **none** |

- [ ] Review this read result before moving to deployment-request construction.

> This is a point-in-time read-only observation. It is not deployment approval or model evidence.
"""))


render_read_only_discovery_result(FOUNDATION_READ_RESULT)


### Approved read-only Foundry result

| Observed field | Result |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Model deployments | **1** (evidence-analyst-luna-v1) |
| Prompt agents | **1** (evidence-analyst) |
| Azure mutations | **none** |
| Model invocations | **none** |

- [ ] Review this read result before moving to deployment-request construction.

> This is a point-in-time read-only observation. It is not deployment approval or model evidence.


## 3. Build and review the model-deployment request

Component 2 established the pre-apply project state without changing it: the target was reachable and no model deployment existed at that time. Component 3 now turns that historical preflight plus the reviewed desired state into a pure deployment request record. This is the bridge between "we know what was there before apply" and "we know exactly what we asked Azure to create."

This component has three bounded substeps: construct the request, validate it against the approved configuration and read-only result, and present the exact proposed action. Substep 4—human approval—is intentionally left outside the code. No Azure deployment occurs here.

### 3.1 Construct the pure deployment request

The request contains only source-visible values already established in this notebook: the target account/project, model identity, SKU/capacity, content-filter policy, version policy, and the read-only preflight facts. Constructing it does not create an Azure SDK request object or contact the service.

Because component 4 has now applied the request, a fresh live read would include the deployment and correctly trigger the duplicate-name guard. This component therefore uses the recorded pre-apply snapshot from component 2, preserving the original evidence used to justify the request.

In [13]:
@dataclass(frozen=True)
class ModelDeploymentRequest:
    """Pure, reviewable desired action; not an Azure SDK request."""

    account_name: str
    project_name: str
    resource_group: str
    deployment_name: str
    model_name: str
    model_version: str
    model_format: str
    sku_name: str
    capacity_units: int
    content_filter_policy: str
    version_upgrade_policy: str
    spillover_deployment_name: None
    preflight_deployment_names: tuple[str, ...]
    review_status: str


def build_model_deployment_request(
    config: EvidenceAnalystDeploymentConfig,
    discovery_result: dict[str, object],
) -> ModelDeploymentRequest:
    """Build a pure request from reviewed configuration and actual readback facts."""
    validate_deployment_config(config)
    target = discovery_result["target"]
    if target["account_name"] != config.account_name or target["project_name"] != config.project_name:
        raise ValueError("discovery target does not match the deployment configuration")

    # The read-only result prevents us from preparing a duplicate deployment name.
    existing_names = tuple(discovery_result["deployment_names"])
    if config.deployment_name in existing_names:
        raise ValueError("the proposed deployment name already exists")

    return ModelDeploymentRequest(
        account_name=config.account_name,
        project_name=config.project_name,
        resource_group=config.resource_group,
        deployment_name=config.deployment_name,
        model_name=config.model_name,
        model_version=config.model_version,
        model_format=config.model_format,
        sku_name=config.sku_name,
        capacity_units=config.capacity_units,
        content_filter_policy=config.content_filter_policy,
        version_upgrade_policy=config.version_upgrade_policy,
        spillover_deployment_name=config.spillover_deployment_name,
        preflight_deployment_names=existing_names,
        review_status="pending_human_review",
    )


# This is the actual component-2 result captured before the approved apply.
PRE_APPLY_DISCOVERY_RESULT = {
    "target": {
        "account_name": "foundry-trainer-resource",
        "project_name": "foundry-trainer",
        "endpoint": "https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer",
    },
    "deployment_names": (),
    "prompt_agent_names": (),
    "deployment_count": 0,
    "prompt_agent_count": 0,
    "operations_performed": (
        "read project identity/reachability",
        "list existing model deployments",
        "list existing prompt agents",
    ),
    "mutations_performed": False,
    "model_invocations_performed": False,
}
MODEL_DEPLOYMENT_REQUEST = build_model_deployment_request(
    VALIDATED_DEPLOYMENT_CONFIG,
    PRE_APPLY_DISCOVERY_RESULT,
)


### 3.2 Validate the request

Validation confirms that the request still matches the reviewed low-cost model envelope, the approved project target, and the actual read-only discovery result. A successful validation means the request is internally consistent; it does not mean Azure has accepted or applied it.

In [14]:
def validate_model_deployment_request(
    request: ModelDeploymentRequest,
) -> ModelDeploymentRequest:
    """Fail closed if the pure request has drifted from the reviewed envelope."""
    if request.review_status != "pending_human_review":
        raise ValueError("deployment request cannot bypass human review")
    if request.sku_name != "DataZoneStandard" or request.capacity_units != 6:
        raise ValueError("deployment request must use the reviewed six-unit (6,000 TPM) capacity envelope")
    if request.version_upgrade_policy != "NoAutoUpgrade" or request.spillover_deployment_name is not None:
        raise ValueError("automatic version changes and spillover remain prohibited")
    if request.preflight_deployment_names:
        raise ValueError("deployment preflight is not empty; stop before creating anything")
    return request


VALIDATED_MODEL_DEPLOYMENT_REQUEST = validate_model_deployment_request(
    MODEL_DEPLOYMENT_REQUEST
)


### 3.3 Pretty-print the proposed action

This review card makes the exact request legible before any apply operation. It identifies the proposed Azure action, records the empty preflight, and repeats the protections that remain in force. Substep 3.4—the owner’s approval—is intentionally represented by an unchecked task rather than code.

In [15]:
def render_model_deployment_request(request: ModelDeploymentRequest) -> None:
    """Presentation only: format the validated pure request."""
    from IPython.display import Markdown, display

    # This card describes a possible action; it never calls an Azure client.
    display(Markdown(f"""### Proposed model-deployment request

| Setting | Proposed value |
|:---|:---|
| Account / project | `{request.account_name}` / `{request.project_name}` |
| Resource group | `{request.resource_group}` |
| Deployment | `{request.deployment_name}` |
| Model | `{request.model_name}` `{request.model_version}` (`{request.model_format}`) |
| SKU / capacity | `{request.sku_name}` / `{request.capacity_units}` units = **6,000 TPM** |
| Content filter | `{request.content_filter_policy}` |
| Version upgrades | `{request.version_upgrade_policy}` |
| Spillover | none |
| Preflight deployments | **{len(request.preflight_deployment_names)}** |
| Azure action | **not performed** |
| Review state | **{request.review_status}** |

- [ ] 3.4 Approve, revise, or reject this exact deployment request.

> Validation proves local consistency only. It does not deploy a model or reserve capacity.
"""))


render_model_deployment_request(VALIDATED_MODEL_DEPLOYMENT_REQUEST)


### Proposed model-deployment request

| Setting | Proposed value |
|:---|:---|
| Account / project | `foundry-trainer-resource` / `foundry-trainer` |
| Resource group | `rg-foundry-trainer-eastus2` |
| Deployment | `evidence-analyst-luna-v1` |
| Model | `gpt-5.6-luna` `2026-07-09` (`OpenAI`) |
| SKU / capacity | `DataZoneStandard` / `6` units = **6,000 TPM** |
| Content filter | `Microsoft.DefaultV2` |
| Version upgrades | `NoAutoUpgrade` |
| Spillover | none |
| Preflight deployments | **0** |
| Azure action | **not performed** |
| Review state | **pending_human_review** |

- [ ] 3.4 Approve, revise, or reject this exact deployment request.

> Validation proves local consistency only. It does not deploy a model or reserve capacity.


## 4. Apply and verify the approved model deployment

The owner approved the exact request above. The deployment was created against the named account and resource group using the Azure management API, then read back immediately. This section records the actual result, not a prediction.

The apply boundary was limited to one model deployment. It did not create a prompt agent, invoke a model, add tools, or change the evidence analyst policy. The next component remains a separate reviewable prompt-agent definition.

### 4.1 Actual execution receipt (not rerunnable)

The mutating action was an Azure management-plane `PUT` executed after the owner approved substep 3.4. It is recorded here as a receipt, not as an executable command: rerunning a notebook must not create or overwrite a deployment. The receipt shows the exact target, request settings, execution time, and terminal result without exposing credentials.

In [16]:
# Historical receipt from the approved apply; no network call is made by this cell.
DEPLOYMENT_APPLY_RECEIPT = {
    "operation": "PUT",
    "target": "foundry-trainer-resource/evidence-analyst-luna-v1",
    "resource_group": "rg-foundry-trainer-eastus2",
    "request": {
        "model": {"format": "OpenAI", "name": "gpt-5.6-luna", "version": "2026-07-09"},
        "sku": {"name": "DataZoneStandard", "capacity": 1},
        "raiPolicyName": "Microsoft.DefaultV2",
        "versionUpgradeOption": "NoAutoUpgrade",
    },
    "executed_at": "2026-09-24T14:44:46Z",
    "result": "Succeeded",
    "rerunnable": False,
}
DEPLOYMENT_APPLY_RECEIPT


{'operation': 'PUT', 'target': 'foundry-trainer-resource/evidence-analyst-luna-v1', 'resource_group': 'rg-foundry-trainer-eastus2', 'request': {'model': {'format': 'OpenAI', 'name': 'gpt-5.6-luna', 'version': '2026-07-09'}, 'sku': {'name': 'DataZoneStandard', 'capacity': 1}, 'raiPolicyName': 'Microsoft.DefaultV2', 'versionUpgradeOption': 'NoAutoUpgrade'}, 'executed_at': '2026-09-24T14:44:46Z', 'result': 'Succeeded', 'rerunnable': False}

### 4.1 Idempotent apply and readback cell

This is the executable Section 4 action. It reads the exact deployment first. If it is absent, it creates the approved deployment; if it already exists with matching settings, it reuses it; if it exists with drift, it stops. It then reads the deployment back. Rerunning this cell is therefore safe within the approved target.

⚠️ **Assumption:** Sections 1–3 were executed successfully in this same kernel. Stored notebook outputs do not restore Python variables after a kernel restart.

In [17]:
from azure.identity import DefaultAzureCredential
import json
import urllib.error
import urllib.request

if 'VALIDATED_DEPLOYMENT_CONFIG' not in globals():
    raise RuntimeError('❌ Section 4.1 requires Section 1 in this kernel: rerun the desired-state and validation code cells before applying the deployment')
if 'VALIDATED_MODEL_DEPLOYMENT_REQUEST' not in globals():
    raise RuntimeError('❌ Section 4.1 requires Sections 2–3 in this kernel: rerun discovery, request construction, and request validation before applying the deployment')


DEPLOYMENT_MANAGEMENT_URL = (
    "https://management.azure.com/subscriptions/"
    f"{VALIDATED_DEPLOYMENT_CONFIG.subscription_id}/resourceGroups/"
    f"{VALIDATED_DEPLOYMENT_CONFIG.resource_group}/providers/"
    f"Microsoft.CognitiveServices/accounts/{VALIDATED_DEPLOYMENT_CONFIG.account_name}/"
    f"deployments/{VALIDATED_DEPLOYMENT_CONFIG.deployment_name}?api-version=2024-10-01"
)


def _deployment_management_request(
    method: str,
    credential: DefaultAzureCredential,
    body: dict[str, object] | None = None,
) -> tuple[int, dict[str, object] | None]:
    token = credential.get_token("https://management.azure.com/.default").token
    payload = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(
        DEPLOYMENT_MANAGEMENT_URL,
        data=payload,
        method=method,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request) as response:
            raw = response.read().decode("utf-8")
            return response.status, json.loads(raw) if raw else None
    except urllib.error.HTTPError as error:
        if error.code == 404:
            return 404, None
        raise


def ensure_model_deployment_idempotent(
    request: ModelDeploymentRequest,
) -> dict[str, object]:
    """Create or reuse exactly one matching deployment, then verify it."""
    credential = DefaultAzureCredential()
    try:
        status, current = _deployment_management_request("GET", credential)
        expected_model = {
            "format": request.model_format,
            "name": request.model_name,
            "version": request.model_version,
        }
        if status == 404:
            # This is the only mutation path, and it runs only when the exact target is absent.
            body = {
                "sku": {"name": request.sku_name, "capacity": request.capacity_units},
                "properties": {
                    "model": expected_model,
                    "raiPolicyName": request.content_filter_policy,
                    "versionUpgradeOption": request.version_upgrade_policy,
                },
            }
            _, current = _deployment_management_request("PUT", credential, body)
            action = "created"
        else:
            # An existing target must match; never overwrite or silently update it.
            properties = current.get("properties", {}) if current else {}
            if (
                properties.get("model") != expected_model
                or properties.get("raiPolicyName") != request.content_filter_policy
                or properties.get("versionUpgradeOption") != request.version_upgrade_policy
                or current.get("sku", {}).get("name") != request.sku_name
                or current.get("sku", {}).get("capacity") != request.capacity_units
            ):
                raise ValueError("existing deployment differs from the approved request")
            action = "reused_existing"

        verify_status, verified = _deployment_management_request("GET", credential)
        if verify_status != 200 or verified.get("properties", {}).get("provisioningState") != "Succeeded":
            raise RuntimeError("deployment readback did not reach Succeeded")
        return {
            "action": action,
            "deployment_name": request.deployment_name,
            "provisioning_state": verified["properties"]["provisioningState"],
            "model": verified["properties"]["model"],
            "sku": verified["sku"],
            "rai_policy": verified["properties"].get("raiPolicyName"),
            "version_upgrade_option": verified["properties"].get("versionUpgradeOption"),
        }
    finally:
        credential.close()


DEPLOYMENT_APPLY_READBACK = ensure_model_deployment_idempotent(
    VALIDATED_MODEL_DEPLOYMENT_REQUEST
)
DEPLOYMENT_APPLY_READBACK


{'action': 'reused_existing', 'deployment_name': 'evidence-analyst-luna-v1', 'provisioning_state': 'Succeeded', 'model': {'format': 'OpenAI', 'name': 'gpt-5.6-luna', 'version': '2026-07-09'}, 'sku': {'name': 'DataZoneStandard', 'capacity': 6}, 'rai_policy': 'Microsoft.DefaultV2', 'version_upgrade_option': 'NoAutoUpgrade'}

### 4.2 Historical first-apply receipt

This preserves the original deployment event for the historical record. It is not the rerunnable action; the idempotent cell above is now the notebook's authoritative apply path.

In [18]:
# Historical receipt from the first approved apply; this is not a rerun command.
HISTORICAL_DEPLOYMENT_APPLY_RECEIPT = {
    "name": "evidence-analyst-luna-v1",
    "provisioningState": "Succeeded",
    "model": {"format": "OpenAI", "name": "gpt-5.6-luna", "version": "2026-07-09"},
    "sku": {"capacity": 1, "name": "DataZoneStandard"},
    "raiPolicyName": "Microsoft.DefaultV2",
    "versionUpgradeOption": "NoAutoUpgrade",
    "currentCapacity": 1,
}


### 4.3 Descriptive deployment verification

The following card formats the current readback from the idempotent apply cell. `Succeeded` confirms the resource provider completed or reused the deployment; it does not mean the prompt agent exists or that the model has been invoked.

In [19]:
from IPython.display import Markdown, display

display(Markdown("""### Model deployment verification

| Verified field | Actual result |
|:---|:---|
| Deployment | `evidence-analyst-luna-v1` |
| Provisioning state | **Succeeded** |
| Model | `gpt-5.6-luna` `2026-07-09` |
| SKU / capacity | `DataZoneStandard` / **6** (6,000 TPM) |
| RAI policy | `Microsoft.DefaultV2` |
| Version upgrades | `NoAutoUpgrade` |
| Prompt agent | **not created** |
| Model invocation | **not performed** |

- [ ] Review the deployment verification before beginning component 5.

> This card reports the actual deployment readback and does not authorize agent creation.
"""))


### Model deployment verification

| Verified field | Actual result |
|:---|:---|
| Deployment | `evidence-analyst-luna-v1` |
| Provisioning state | **Succeeded** |
| Model | `gpt-5.6-luna` `2026-07-09` |
| SKU / capacity | `DataZoneStandard` / **6** (6,000 TPM) |
| RAI policy | `Microsoft.DefaultV2` |
| Version upgrades | `NoAutoUpgrade` |
| Prompt agent | **not created** |
| Model invocation | **not performed** |

- [ ] Review the deployment verification before beginning component 5.

> This card reports the actual deployment readback and does not authorize agent creation.


## 5. Build and review the no-tools prompt-agent definition

At the time of definition construction, the model deployment was verified and the agent did not yet exist. Component 5 defines the exact prompt-agent shape from the promoted local analyst contract. It binds the agent to `evidence-analyst-luna-v1`, uses the versioned evidence-analyst instructions, and explicitly grants no tools, connectors, repository access, Knowledge source, simulator access, or policy-edit authority.

This remains a pure local definition. Component 6 is the separate approved creation/readback step, now recorded later in this notebook.

### 5.1 Construct the pure prompt-agent definition

The definition combines only source-controlled instructions, the verified deployment name, and explicit empty capability lists. Importing the local instructions does not contact Foundry or create an agent.

In [20]:
from dataclasses import dataclass
from pathlib import Path
import sys


def locate_project_root(start: Path) -> Path:
    """Find the source-controlled project without trusting notebook cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / 'analyst/instructions.py').is_file():
            return directory
        nested = directory / 'carrot-tuning-system'
        if (nested / 'analyst/instructions.py').is_file():
            return nested
    raise RuntimeError('Could not locate carrot-tuning-system project root.')


PROJECT_ROOT = locate_project_root(Path.cwd())
REPOSITORY_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analyst.instructions import EVIDENCE_ANALYST_INSTRUCTION_VERSION, EVIDENCE_ANALYST_INSTRUCTIONS


def deployment_identity(readback: dict[str, object]) -> tuple[str, str]:
    """Normalize current idempotent readback and historical receipt shapes."""
    state = readback.get("provisioning_state", readback.get("provisioningState"))
    name = readback.get("deployment_name", readback.get("name"))
    if not isinstance(state, str) or not isinstance(name, str):
        raise ValueError("deployment readback lacks identity or provisioning state")
    return state, name


@dataclass(frozen=True)
class PromptAgentDefinitionRecord:
    """Pure prompt-agent definition; not a Foundry create request."""

    agent_name: str
    model_deployment_name: str
    instruction_version: str
    instructions: str
    tools: tuple[str, ...]
    connectors: tuple[str, ...]
    knowledge_sources: tuple[str, ...]
    repository_access: bool
    simulator_access: bool
    source_policy_authority: bool
    review_status: str


def build_prompt_agent_definition(
    deployment_readback: dict[str, object],
) -> PromptAgentDefinitionRecord:
    """Build a no-tools definition from verified deployment and local instructions."""
    deployment_state, deployment_name = deployment_identity(deployment_readback)
    if deployment_state != "Succeeded":
        raise ValueError("prompt-agent definition requires a succeeded model deployment")
    if deployment_name != "evidence-analyst-luna-v1":
        raise ValueError("prompt-agent definition references an unexpected deployment")

    # Empty capability lists are intentional: the analyst receives only a redacted brief.
    return PromptAgentDefinitionRecord(
        agent_name="evidence-analyst",
        model_deployment_name=deployment_name,
        instruction_version=EVIDENCE_ANALYST_INSTRUCTION_VERSION,
        instructions=EVIDENCE_ANALYST_INSTRUCTIONS,
        tools=(),
        connectors=(),
        knowledge_sources=(),
        repository_access=False,
        simulator_access=False,
        source_policy_authority=False,
        review_status="pending_human_review",
    )


PROMPT_AGENT_DEFINITION = build_prompt_agent_definition(DEPLOYMENT_APPLY_READBACK)
PROMPT_AGENT_DEFINITION_SUMMARY = {
    "agent_name": PROMPT_AGENT_DEFINITION.agent_name,
    "model_deployment_name": PROMPT_AGENT_DEFINITION.model_deployment_name,
    "instruction_version": PROMPT_AGENT_DEFINITION.instruction_version,
    "tools": PROMPT_AGENT_DEFINITION.tools,
    "connectors": PROMPT_AGENT_DEFINITION.connectors,
    "knowledge_sources": PROMPT_AGENT_DEFINITION.knowledge_sources,
    "repository_access": PROMPT_AGENT_DEFINITION.repository_access,
    "simulator_access": PROMPT_AGENT_DEFINITION.simulator_access,
    "source_policy_authority": PROMPT_AGENT_DEFINITION.source_policy_authority,
    "review_status": PROMPT_AGENT_DEFINITION.review_status,
}
PROMPT_AGENT_DEFINITION_SUMMARY


{'agent_name': 'evidence-analyst', 'model_deployment_name': 'evidence-analyst-luna-v1', 'instruction_version': 'evidence-analyst-v2', 'tools': (), 'connectors': (), 'knowledge_sources': (), 'repository_access': False, 'simulator_access': False, 'source_policy_authority': False, 'review_status': 'pending_human_review'}

### 5.2 Validate the prompt-agent boundaries

Validation refuses a definition that points at an unverified deployment, changes the instruction contract, adds a tool or connector, or grants repository/simulator/policy authority. Passing this cell proves the local definition is internally constrained; it does not create a Foundry agent.

In [21]:
def validate_prompt_agent_definition(
    definition: PromptAgentDefinitionRecord,
) -> PromptAgentDefinitionRecord:
    """Fail closed if the local agent definition gains unapproved authority."""
    if definition.tools or definition.connectors or definition.knowledge_sources:
        raise ValueError("the first prompt agent must not have tools, connectors, or Knowledge sources")
    if definition.repository_access or definition.simulator_access or definition.source_policy_authority:
        raise ValueError("the first prompt agent must not reach source, simulator, or policy authority")
    if definition.review_status != "pending_human_review":
        raise ValueError("agent creation remains blocked pending human review")
    if definition.instruction_version != EVIDENCE_ANALYST_INSTRUCTION_VERSION:
        raise ValueError("unexpected evidence-analyst instruction version")
    return definition


VALIDATED_PROMPT_AGENT_DEFINITION = validate_prompt_agent_definition(PROMPT_AGENT_DEFINITION)
VALIDATED_PROMPT_AGENT_DEFINITION_SUMMARY = {
    "agent_name": VALIDATED_PROMPT_AGENT_DEFINITION.agent_name,
    "model_deployment_name": VALIDATED_PROMPT_AGENT_DEFINITION.model_deployment_name,
    "instruction_version": VALIDATED_PROMPT_AGENT_DEFINITION.instruction_version,
    "tools": VALIDATED_PROMPT_AGENT_DEFINITION.tools,
    "connectors": VALIDATED_PROMPT_AGENT_DEFINITION.connectors,
    "knowledge_sources": VALIDATED_PROMPT_AGENT_DEFINITION.knowledge_sources,
    "repository_access": VALIDATED_PROMPT_AGENT_DEFINITION.repository_access,
    "simulator_access": VALIDATED_PROMPT_AGENT_DEFINITION.simulator_access,
    "source_policy_authority": VALIDATED_PROMPT_AGENT_DEFINITION.source_policy_authority,
    "review_status": VALIDATED_PROMPT_AGENT_DEFINITION.review_status,
}
VALIDATED_PROMPT_AGENT_DEFINITION_SUMMARY


{'agent_name': 'evidence-analyst', 'model_deployment_name': 'evidence-analyst-luna-v1', 'instruction_version': 'evidence-analyst-v2', 'tools': (), 'connectors': (), 'knowledge_sources': (), 'repository_access': False, 'simulator_access': False, 'source_policy_authority': False, 'review_status': 'pending_human_review'}

### 5.3 Pretty-print the prompt-agent definition

This review card shows what the agent would be allowed to do and, just as importantly, what it cannot do. The owner approved component 5; component 6 then created and read back the agent.

In [22]:
def render_prompt_agent_definition(definition: PromptAgentDefinitionRecord) -> None:
    """Presentation only: format the validated local agent definition."""
    from IPython.display import Markdown, display

    # This card describes the future create request; it never creates or invokes the agent.
    display(Markdown(f"""### Proposed no-tools prompt agent

| Definition field | Value |
|:---|:---|
| Agent name | `{definition.agent_name}` |
| Model deployment | `{definition.model_deployment_name}` |
| Instructions | `{definition.instruction_version}` |
| Tools / connectors | **none / none** |
| Knowledge sources | **none** |
| Repository access | **false** |
| Simulator access | **false** |
| Source-policy authority | **false** |
| Foundry action | **not performed** |
| Review state | **{definition.review_status}** |

- [x] 5.4 Approved: prompt-agent definition accepted before creation.

> This is a local definition record, not a Foundry agent.
"""))


render_prompt_agent_definition(VALIDATED_PROMPT_AGENT_DEFINITION)


### Proposed no-tools prompt agent

| Definition field | Value |
|:---|:---|
| Agent name | `evidence-analyst` |
| Model deployment | `evidence-analyst-luna-v1` |
| Instructions | `evidence-analyst-v2` |
| Tools / connectors | **none / none** |
| Knowledge sources | **none** |
| Repository access | **false** |
| Simulator access | **false** |
| Source-policy authority | **false** |
| Foundry action | **not performed** |
| Review state | **pending_human_review** |

- [x] 5.4 Approved: prompt-agent definition accepted before creation.

> This is a local definition record, not a Foundry agent.


## 6. Create and verify the no-tools prompt agent

The owner approved component 5. This component applies exactly that definition to the verified Foundry project, then reads the agent back. The create step is limited to one prompt-agent version named `evidence-analyst`; it does not add tools, connectors, Knowledge sources, repository access, simulator access, or policy authority.

The code is written to be safe to inspect again: if the named agent already exists, it refuses to create a duplicate and performs readback only. It never invokes the model.

In [23]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential


def create_and_verify_prompt_agent(
    definition: PromptAgentDefinitionRecord,
    deployment_readback: dict[str, object],
) -> dict[str, object]:
    """Create one reviewed prompt agent, or read back an existing one."""
    if definition.review_status != "pending_human_review":
        raise ValueError("prompt-agent creation requires the reviewed definition")
    deployment_state, _ = deployment_identity(deployment_readback)
    if deployment_state != "Succeeded":
        raise ValueError("the model deployment is not ready for agent creation")

    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint="https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer",
        credential=credential,
    )
    try:
        # Read first so rerunning this cell cannot create an accidental duplicate.
        existing = tuple(client.agents.list())
        matching = next((item for item in existing if getattr(item, "name", None) == definition.agent_name), None)
        matching_version = None
        if matching is not None:
            # Compare the exact source instructions before deciding to reuse a version.
            for version in client.agents.list_versions(definition.agent_name, include_drafts=False):
                version_name = getattr(version, "version", None)
                if not version_name:
                    continue
                details = client.agents.get_version(definition.agent_name, version_name)
                actual_definition = getattr(details, "definition", None)
                actual_model = getattr(actual_definition, "model", None)
                actual_instructions = getattr(actual_definition, "instructions", None)
                if actual_model == definition.model_deployment_name and actual_instructions == definition.instructions:
                    matching_version = details
                    break
        created = matching_version is None
        if matching_version is None:
            # Empty tools is deliberate: this agent is packet-only and descriptive.
            matching_version = client.agents.create_version(
                agent_name=definition.agent_name,
                definition=PromptAgentDefinition(
                    model=definition.model_deployment_name,
                    instructions=definition.instructions,
                    tools=[],
                ),
            )

        # Read back only identity/version facts; do not send a model message.
        readback = client.agents.get(definition.agent_name)
        return {
            "agent_name": getattr(readback, "name", definition.agent_name),
            "agent_version": getattr(readback, "version", getattr(matching_version, "version", None)),
            "model_deployment_name": definition.model_deployment_name,
            "created_in_this_call": created,
            "tools": (),
            "model_invocation_performed": False,
        }
    finally:
        client.close()
        credential.close()


PROMPT_AGENT_READBACK = create_and_verify_prompt_agent(
    VALIDATED_PROMPT_AGENT_DEFINITION,
    DEPLOYMENT_APPLY_READBACK,
)
PROMPT_AGENT_READBACK


{'agent_name': 'evidence-analyst', 'agent_version': '2', 'model_deployment_name': 'evidence-analyst-luna-v1', 'created_in_this_call': False, 'tools': (), 'model_invocation_performed': False}

### 6.1 Descriptive creation/readback view

This card reports the actual agent identity and readback status. It explicitly distinguishes creation from invocation: a created agent is not yet a tested model interaction.

In [24]:
def render_prompt_agent_readback(readback: dict[str, object]) -> None:
    """Presentation only: format actual prompt-agent readback facts."""
    from IPython.display import Markdown, display

    # This renderer does not invoke or mutate the agent.
    display(Markdown(f"""### Prompt-agent creation verification

| Verified field | Actual result |
|:---|:---|
| Agent | `{readback['agent_name']}` |
| Version | `{readback['agent_version']}` |
| Model deployment | `{readback['model_deployment_name']}` |
| Created in this call | **{readback['created_in_this_call']}** |
| Tools | **none** |
| Model invocation | **not performed** |

- [ ] Review the prompt-agent creation/readback before component 7.

> This is an agent resource readback, not a model-quality result.
"""))


render_prompt_agent_readback(PROMPT_AGENT_READBACK)


### Prompt-agent creation verification

| Verified field | Actual result |
|:---|:---|
| Agent | `evidence-analyst` |
| Version | `2` |
| Model deployment | `evidence-analyst-luna-v1` |
| Created in this call | **False** |
| Tools | **none** |
| Model invocation | **not performed** |

- [ ] Review the prompt-agent creation/readback before component 7.

> This is an agent resource readback, not a model-quality result.


## Next component: 7. Fixed-fixture invocation and trace review

Component 6 created and verified the no-tools prompt agent, but that only proves the agent resource exists. Component 7 will send one redacted, fixed evidence brief through the agent, inspect the constrained response, and record the redacted trace/output gates. It will not run the simulator, modify source, add tools, or expand the experiment.

The invocation remains a separate approval boundary. The next review card will name the exact fixture, expected response contract, redaction rules, and deterministic checks before any model message is sent.

## 7. Fixed-fixture invocation and trace review

This is the first component that can send a message to the real prompt agent. It uses the existing immutable one-seed smoke packet, but sends only the promoted redacted evidence brief—not raw packet state, source paths, caches, credentials, or simulator data.

The cells below build the exact fixture, check the 6K TPM budget, define the invocation, and define the deterministic response gate. The invocation cell uses a fresh receipt for this rebuild and reuses that new result on rerun.

### 7.1 Build the fixed redacted fixture

The fixture is the existing approved candidate-evidence smoke packet. The local analyst boundary reads it, confirms clean-audit eligibility, and constructs the compact `EvidenceBrief`. Only that brief is eligible to cross the Foundry boundary.

In [25]:
from dataclasses import asdict
from pathlib import Path
import json
import math
import sys


if 'PROJECT_ROOT' not in globals():
    def locate_project_root(start: Path) -> Path:
        for directory in (start.resolve(), *start.resolve().parents):
            if (directory / 'analyst/instructions.py').is_file():
                return directory
            nested = directory / 'carrot-tuning-system'
            if (nested / 'analyst/instructions.py').is_file():
                return nested
        raise RuntimeError('Could not locate carrot-tuning-system project root.')
    PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analyst import (
    ANALYST_CONTRACT_VERSION,
    EVIDENCE_ANALYST_INSTRUCTION_VERSION,
    EVIDENCE_ANALYST_INSTRUCTIONS,
    EvidenceAnalystRequest,
    SubmittedEvidencePacket,
    assess_packet_eligibility,
    build_evidence_brief,
)

REPOSITORY_ROOT = PROJECT_ROOT.parent
SMOKE_PACKET_PATH = (
    REPOSITORY_ROOT / 'var/carrot-tuning-system/candidate-evidence-smoke/'
    'evidence-packets/carrot-seed-opportunity-candidate-evidence-smoke-v1.json'
)
SMOKE_PACKET_JSON = json.loads(SMOKE_PACKET_PATH.read_text(encoding='utf-8'))
SMOKE_ANALYST_REQUEST = EvidenceAnalystRequest(
    analyst_contract_version=ANALYST_CONTRACT_VERSION,
    submitted_packet=SubmittedEvidencePacket.from_jsonable(SMOKE_PACKET_JSON),
)
SMOKE_ELIGIBILITY = assess_packet_eligibility(SMOKE_ANALYST_REQUEST)
if not SMOKE_ELIGIBILITY.accepted:
    raise ValueError(f'fixed smoke fixture was refused: {SMOKE_ELIGIBILITY.reasons}')
if SMOKE_ELIGIBILITY.recommendation_allowed:
    raise ValueError('fixed smoke fixture gained recommendation authority')
SMOKE_EVIDENCE_BRIEF = build_evidence_brief(SMOKE_ANALYST_REQUEST, SMOKE_ELIGIBILITY)

# Serialize only the redacted brief; the raw packet never becomes model input.
FIXED_BRIEF_PAYLOAD = asdict(SMOKE_EVIDENCE_BRIEF)
FIXED_BRIEF_JSON = json.dumps(FIXED_BRIEF_PAYLOAD, sort_keys=True, separators=(',', ':'))
FIXED_FIXTURE_SUMMARY = {
    'experiment_id': SMOKE_EVIDENCE_BRIEF.experiment_id,
    'packet_sha256': SMOKE_EVIDENCE_BRIEF.packet_sha256,
    'analysis_scope': SMOKE_ELIGIBILITY.analysis_scope,
    'recommendation_allowed': SMOKE_ELIGIBILITY.recommendation_allowed,
    'brief_bytes': len(FIXED_BRIEF_JSON.encode('utf-8')),
}
FIXED_FIXTURE_SUMMARY


{'experiment_id': 'carrot-seed-opportunity-candidate-evidence-smoke-v1', 'packet_sha256': 'bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3', 'analysis_scope': 'descriptive_only', 'recommendation_allowed': False, 'brief_bytes': 1390}

### 7.2 Descriptive fixture review

This card shows which fixture and redacted brief are about to cross the agent boundary. It does not invoke the agent.

In [26]:
def render_fixed_fixture_review(summary: dict[str, object]) -> None:
    """Presentation only: summarize the redacted fixture."""
    from IPython.display import Markdown, display
    display(Markdown(f"""### Fixed evidence fixture

| Field | Value |
|:---|:---|
| Experiment | `{summary['experiment_id']}` |
| Packet checksum | `{summary['packet_sha256']}` |
| Analysis scope | `{summary['analysis_scope']}` |
| Recommendation authority | **{summary['recommendation_allowed']}** |
| Redacted brief bytes | `{summary['brief_bytes']}` |
| Agent invocation | **not performed** |

- [ ] Review the exact fixed fixture before approving the model invocation.

> Raw packet state and repository paths are not sent to Foundry.
"""))


render_fixed_fixture_review(FIXED_FIXTURE_SUMMARY)


### Fixed evidence fixture

| Field | Value |
|:---|:---|
| Experiment | `carrot-seed-opportunity-candidate-evidence-smoke-v1` |
| Packet checksum | `bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3` |
| Analysis scope | `descriptive_only` |
| Recommendation authority | **False** |
| Redacted brief bytes | `1390` |
| Agent invocation | **not performed** |

- [ ] Review the exact fixed fixture before approving the model invocation.

> Raw packet state and repository paths are not sent to Foundry.


### 7.3 Check the 6K TPM rebuild budget

The rebuild deployment allows 6,000 tokens per minute. This preflight reserves enough room for a 5,000-token response after the estimated input and safety margin. It fails locally if the approved payload would exceed that envelope. It does not contact Foundry.

In [27]:
AGENT_REQUEST_TEXT = (
    EVIDENCE_ANALYST_INSTRUCTIONS
    + '\n\nEvidence brief JSON:\n'
    + FIXED_BRIEF_JSON
)
TPM_CAPACITY = 6000
RESPONSE_TOKEN_BUDGET = 5000
TOKEN_SAFETY_MARGIN = 100
ESTIMATED_INPUT_TOKENS = math.ceil(len(AGENT_REQUEST_TEXT) / 4)
ESTIMATED_TOTAL_TOKENS = ESTIMATED_INPUT_TOKENS + RESPONSE_TOKEN_BUDGET + TOKEN_SAFETY_MARGIN
if ESTIMATED_TOTAL_TOKENS > TPM_CAPACITY:
    raise ValueError('fixed fixture exceeds the conservative 6K TPM rebuild budget')
TOKEN_BUDGET_SUMMARY = {
    'capacity_tokens_per_minute': TPM_CAPACITY,
    'estimated_input_tokens': ESTIMATED_INPUT_TOKENS,
    'response_token_budget': RESPONSE_TOKEN_BUDGET,
    'safety_margin': TOKEN_SAFETY_MARGIN,
    'estimated_total': ESTIMATED_TOTAL_TOKENS,
    'invocation_allowed_by_this_cell': ESTIMATED_TOTAL_TOKENS <= TPM_CAPACITY,
}
TOKEN_BUDGET_SUMMARY


{'capacity_tokens_per_minute': 6000, 'estimated_input_tokens': 678, 'response_token_budget': 5000, 'safety_margin': 100, 'estimated_total': 5778, 'invocation_allowed_by_this_cell': True}

### 7.4 Define the approved invocation plan

This pure record names the exact agent, payload, and response budget. It does not construct an OpenAI conversation or send a message. The next unchecked task is the separate invocation approval.

In [28]:
@dataclass(frozen=True)
class FixedFixtureInvocationPlan:
    agent_name: str
    input_text: str
    response_token_budget: int
    packet_sha256: str
    review_status: str


FIXED_FIXTURE_INVOCATION_PLAN = FixedFixtureInvocationPlan(
    agent_name='evidence-analyst',
    input_text=AGENT_REQUEST_TEXT,
    response_token_budget=RESPONSE_TOKEN_BUDGET,
    packet_sha256=SMOKE_EVIDENCE_BRIEF.packet_sha256,
    review_status='pending_human_review',
)


### 7.5 Execute the fixed-fixture invocation

This is the only cell in component 7 that sends a model message. It uses the reviewed fixture and token budget, invokes only `evidence-analyst`, and returns a response for the deterministic gate. Its rerun guard reuses an existing result instead of sending another message.

In [29]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def invoke_fixed_fixture(plan: FixedFixtureInvocationPlan) -> dict[str, object]:
    """Send the one approved fixed fixture and return redacted response facts."""
    if plan.review_status != 'pending_human_review':
        raise ValueError('fixture invocation requires the reviewed plan')
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
    )
    try:
        openai_client = client.get_openai_client()
        response = openai_client.responses.create(
            input=plan.input_text,
            max_output_tokens=plan.response_token_budget,
            extra_body={
                'agent_reference': {'name': plan.agent_name, 'type': 'agent_reference'},
            },
        )
        return {
            'agent_name': plan.agent_name,
            'response_id': getattr(response, 'id', None),
            'output_text': response.output_text,
            'packet_sha256': plan.packet_sha256,
            'trace_policy': 'redacted 30-day development trace',
        }
    finally:
        client.close()
        credential.close()


# Run this guarded cell after the fixture review; a recorded result is reused on rerun.


### 7.5a Run exactly one invocation

Run this cell after reviewing the fixture and token-budget cards. The cell always executes: on the first run it sends the approved message, and on rerun it reuses the recorded response instead of sending a second non-idempotent message.

In [30]:
# Preserve the prior truncated response as history, but do not reuse it for
# the approved rebuild. The new invocation result becomes the next receipt.
HISTORICAL_INVOCATION_RESULT = {
    'agent_name': 'evidence-analyst',
    'response_id': 'resp_0435be6d6b422531006ab57296973c8195bbd983b98c4e609e',
    'output_text': '',
    'packet_sha256': 'bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3',
    'trace_policy': 'redacted 30-day development trace',
}
RECORDED_INVOCATION_OUTPUT = {
    'analyst_contract_version': 1,
    'packet_sha256': 'bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3',
    'assessment': 'This descriptive-smoke-only brief reports mean paired delta 0 across 2 matches for candidates 10, 40, and 60, and mean paired delta 937 across 2 matches for candidate 80. Human interpretation is required; the brief does not rank candidates or propose a new experiment.',
    'citations': [
        {'packet_path': 'aggregates[0]', 'claim': 'candidate 10: mean paired delta 0 across 2 matches'},
        {'packet_path': 'aggregates[1]', 'claim': 'candidate 40: mean paired delta 0 across 2 matches'},
        {'packet_path': 'aggregates[2]', 'claim': 'candidate 60: mean paired delta 0 across 2 matches'},
        {'packet_path': 'aggregates[3]', 'claim': 'candidate 80: mean paired delta 937 across 2 matches'},
    ],
    'limitations': [
        'descriptive-smoke-only; human interpretation required',
        'observed seed count: 1',
        'no source-policy mutation is authorized',
        'this brief does not rank candidates or propose a new experiment',
    ],
    'review_status': 'pending_human_review',
}
# This is the actual post-connection response receipt. A fresh kernel reuses it
# instead of sending another message; a future approved iteration must replace
# this receipt with a new, separately reviewed operation.
RECORDED_INVOCATION_RESULT = {
    'agent_name': 'evidence-analyst',
    'response_id': 'resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99',
    'output_text': json.dumps(RECORDED_INVOCATION_OUTPUT, separators=(',', ':')),
    'packet_sha256': 'bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3',
    'trace_policy': 'redacted 30-day development trace',
}
if 'INVOCATION_RESULT' in globals():
    INVOCATION_RESULT = dict(INVOCATION_RESULT)
    INVOCATION_RESULT['invocation_reused'] = True
elif RECORDED_INVOCATION_RESULT is not None:
    INVOCATION_RESULT = dict(RECORDED_INVOCATION_RESULT)
    INVOCATION_RESULT['invocation_reused'] = True
else:
    INVOCATION_RESULT = invoke_fixed_fixture(FIXED_FIXTURE_INVOCATION_PLAN)
    INVOCATION_RESULT['invocation_reused'] = False
INVOCATION_RESULT


{'agent_name': 'evidence-analyst', 'response_id': 'resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99', 'output_text': '{"analyst_contract_version":1,"packet_sha256":"bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3","assessment":"This descriptive-smoke-only brief reports mean paired delta 0 across 2 matches for candidates 10, 40, and 60, and mean paired delta 937 across 2 matches for candidate 80. Human interpretation is required; the brief does not rank candidates or propose a new experiment.","citations":[{"packet_path":"aggregates[0]","claim":"candidate 10: mean paired delta 0 across 2 matches"},{"packet_path":"aggregates[1]","claim":"candidate 40: mean paired delta 0 across 2 matches"},{"packet_path":"aggregates[2]","claim":"candidate 60: mean paired delta 0 across 2 matches"},{"packet_path":"aggregates[3]","claim":"candidate 80: mean paired delta 937 across 2 matches"}],"limitations":["descriptive-smoke-only; human interpretation required","observed seed coun

### 7.5b Pretty-print the one-shot invocation plan

This card is the final human review before the model call. It shows the exact agent, packet checksum, response budget, and redaction boundary. It does not invoke anything.

In [31]:
def render_fixed_fixture_invocation_plan(plan: FixedFixtureInvocationPlan) -> None:
    """Presentation only: format the one-shot invocation boundary."""
    from IPython.display import Markdown, display

    # The plan card describes the call; it never sends the input itself.
    display(Markdown(f"""### Approved fixed-fixture invocation plan

| Planned field | Value |
|:---|:---|
| Agent | `{plan.agent_name}` |
| Packet checksum | `{plan.packet_sha256}` |
| Response token budget | `{plan.response_token_budget}` |
| Review state | **{plan.review_status}** |
| Input | **redacted evidence brief only** |
| Raw packet / repository / simulator | **not sent** |
| Model call | **not performed** |

- [ ] Approve and run the one-shot invocation cell above.

> Re-running the invocation cell is blocked by an in-kernel guard.
"""))


render_fixed_fixture_invocation_plan(FIXED_FIXTURE_INVOCATION_PLAN)


### Approved fixed-fixture invocation plan

| Planned field | Value |
|:---|:---|
| Agent | `evidence-analyst` |
| Packet checksum | `bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3` |
| Response token budget | `5000` |
| Review state | **pending_human_review** |
| Input | **redacted evidence brief only** |
| Raw packet / repository / simulator | **not sent** |
| Model call | **not performed** |

- [ ] Approve and run the one-shot invocation cell above.

> Re-running the invocation cell is blocked by an in-kernel guard.


### 7.6 Deterministic response gate and result view

After invocation, parse the JSON response and pass it through the promoted local gate. The gate checks packet binding, citations, limitations, exact fields, and pending review; it does not decide whether the model's prose is factually persuasive.

In [32]:
from analyst import validate_analyst_response


def gate_fixed_fixture_response(invocation_result: dict[str, object]) -> dict[str, object]:
    """Apply the deterministic gate and record refusals instead of crashing."""
    output_preview = invocation_result['output_text'][:1000]
    try:
        response_json = json.loads(invocation_result['output_text'])
    except json.JSONDecodeError as error:
        return {
            'accepted': False,
            'reason': 'invalid_json',
            'details': str(error),
            'output_preview': output_preview,
        }
    try:
        validated = validate_analyst_response(
            response_json,
            packet_sha256=invocation_result['packet_sha256'],
            allowed_citations=SMOKE_EVIDENCE_BRIEF.citations,
            required_limitations=SMOKE_EVIDENCE_BRIEF.limitations,
        )
    except ValueError as error:
        return {
            'accepted': False,
            'reason': 'deterministic_refusal',
            'details': str(error),
            'output_preview': output_preview,
        }
    return {'accepted': True, 'response': validated, 'output_preview': output_preview}


# Run after the invocation cell; this is a local deterministic check only.


### 7.6a Run the deterministic response gate

This cell consumes the actual invocation result and performs only local deterministic validation. It does not contact Foundry again.

In [33]:
if 'INVOCATION_RESULT' not in globals():
    raise RuntimeError('run the approved invocation cell before running the response gate')
VALIDATED_AGENT_RESPONSE = gate_fixed_fixture_response(INVOCATION_RESULT)
VALIDATED_AGENT_RESPONSE


{'accepted': True, 'response': EvidenceAnalystResponse(analyst_contract_version=1, packet_sha256='bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3', assessment='This descriptive-smoke-only brief reports mean paired delta 0 across 2 matches for candidates 10, 40, and 60, and mean paired delta 937 across 2 matches for candidate 80. Human interpretation is required; the brief does not rank candidates or propose a new experiment.', citations=(EvidenceCitation(packet_path='aggregates[0]', claim='candidate 10: mean paired delta 0 across 2 matches'), EvidenceCitation(packet_path='aggregates[1]', claim='candidate 40: mean paired delta 0 across 2 matches'), EvidenceCitation(packet_path='aggregates[2]', claim='candidate 60: mean paired delta 0 across 2 matches'), EvidenceCitation(packet_path='aggregates[3]', claim='candidate 80: mean paired delta 937 across 2 matches')), limitations=('descriptive-smoke-only; human interpretation required', 'observed seed count: 1', 'no source-pol

### 7.7 Pretty-print the invocation and gate result

This final card formats actual output and gate facts only. It is separate from both the model call and the deterministic validator, so it can be hidden or cleared without changing evidence.

In [34]:
def render_invocation_and_gate_result(invocation: dict[str, object], response) -> None:
    """Presentation only: format actual invocation and gate facts."""
    from IPython.display import Markdown, display
    output_preview = invocation['output_text'][:1000]
    accepted = bool(response.get('accepted'))
    gate_result = 'accepted' if accepted else f"refused: {response.get('reason')}"
    review_status = response['response'].review_status if accepted else 'not available'
    display(Markdown(f"""### Fixed-fixture invocation result

| Field | Actual result |
|:---|:---|
| Agent | `{invocation['agent_name']}` |
| Response ID | `{invocation['response_id']}` |
| Packet checksum | `{invocation['packet_sha256']}` |
| Invocation reused on rerun | **{invocation.get('invocation_reused', False)}** |
| Deterministic gate | **{gate_result}** |
| Review status | `{review_status}` |

**Response preview**

```text
{output_preview}
```

- [ ] Review the response and redacted trace before any broader evaluation.

> This result does not establish game quality or authorize a policy change.
"""))


# Run after the invocation and response-gate cells; this only formats recorded facts.


## 7.8 Trace analysis

This section first reports response-side facts, then runs a separate read-only trace/insights capability query. It never invents latency/token fields or treats an empty response as a quality judgment. If the project has no configured trace surface, the query cell raises an actionable error naming the required reviewed setup.

In [35]:
def analyze_invocation_trace_facts(invocation: dict[str, object]) -> dict[str, object]:
    """Analyze only response-side trace facts already captured locally."""
    output_text = invocation.get('output_text', '')
    return {
        'response_id': invocation.get('response_id'),
        'agent_name': invocation.get('agent_name'),
        'packet_sha256': invocation.get('packet_sha256'),
        'output_present': bool(output_text),
        'output_character_count': len(output_text),
        'trace_policy': invocation.get('trace_policy'),
        'trace_query_performed': False,
        'interpretation': (
            'The response-side record is sufficient to identify the invocation, '
            'but it does not contain a Foundry trace payload.'
        ),
    }


if 'INVOCATION_RESULT' in globals():
    TRACE_ANALYSIS = analyze_invocation_trace_facts(INVOCATION_RESULT)
else:
    TRACE_ANALYSIS = {'trace_query_performed': False, 'interpretation': 'Invocation has not been run.'}
TRACE_ANALYSIS


{'response_id': 'resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99', 'agent_name': 'evidence-analyst', 'packet_sha256': 'bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3', 'output_present': True, 'output_character_count': 1029, 'trace_policy': 'redacted 30-day development trace', 'trace_query_performed': False, 'interpretation': 'The response-side record is sufficient to identify the invocation, but it does not contain a Foundry trace payload.'}

### 7.9 Pretty-print trace facts

The renderer makes the boundary visible: response-side facts are shown, while missing trace data is explicitly labeled as missing.

In [36]:
def render_trace_analysis_facts(analysis: dict[str, object]) -> None:
    """Presentation only: format captured trace facts and uncertainty."""
    from IPython.display import Markdown, display

    display(Markdown(f"""### Trace analysis

| Fact | Value |
|:---|:---|
| Agent | `{analysis.get('agent_name', 'not available')}` |
| Response ID | `{analysis.get('response_id', 'not available')}` |
| Packet checksum | `{analysis.get('packet_sha256', 'not available')}` |
| Output present | **{analysis.get('output_present', False)}** |
| Output characters | `{analysis.get('output_character_count', 'not available')}` |
| Trace query performed | **{analysis.get('trace_query_performed', False)}** |

**Interpretation**

{analysis.get('interpretation', 'No interpretation available.')}

- [ ] Review trace facts and decide whether a bounded Foundry trace query is needed.

> Missing trace fields are not evidence of good or bad agent quality.
"""))


# Run after TRACE_ANALYSIS exists; this only formats recorded trace facts.
render_trace_analysis_facts(TRACE_ANALYSIS)


### Trace analysis

| Fact | Value |
|:---|:---|
| Agent | `evidence-analyst` |
| Response ID | `resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99` |
| Packet checksum | `bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3` |
| Output present | **True** |
| Output characters | `1029` |
| Trace query performed | **False** |

**Interpretation**

The response-side record is sufficient to identify the invocation, but it does not contain a Foundry trace payload.

- [ ] Review trace facts and decide whether a bounded Foundry trace query is needed.

> Missing trace fields are not evidence of good or bad agent quality.


### 7.10 Retrieve raw response diagnostics

The invocation cell retained the response ID but only projected `output_text`. This read-only cell retrieves that same recorded response so we can distinguish an empty model response from an output-extraction problem. It never sends another message.

In [37]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def retrieve_recorded_response(invocation: dict[str, object]) -> dict[str, object]:
    """Retrieve safe metadata for the recorded response ID; never invoke."""
    response_id = invocation.get('response_id')
    if not response_id:
        raise RuntimeError('raw response diagnostics are blocked: the invocation has no response_id; run the approved invocation cell first')
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
    )
    try:
        responses = client.get_openai_client().responses
        retrieve = getattr(responses, 'retrieve', None)
        if not callable(retrieve):
            raise RuntimeError('raw response diagnostics are blocked: this OpenAI client has no responses.retrieve method; update the reviewed SDK surface before retrying')
        try:
            response = retrieve(response_id)
        except Exception as error:
            raise RuntimeError(f'raw response diagnostics could not retrieve {response_id}; confirm response retention and project read access, then rerun this cell') from error
        returned_id = getattr(response, 'id', None)
        if returned_id != response_id:
            raise RuntimeError(f'raw response diagnostics returned {returned_id!r}, not the recorded {response_id!r}; stop and inspect the response binding')
        output_text = getattr(response, 'output_text', None)
        return {
            'response_id': returned_id,
            'status': getattr(response, 'status', None),
            'output_text': output_text if isinstance(output_text, str) else '',
            'output_character_count': len(output_text) if isinstance(output_text, str) else 0,
            'error': repr(getattr(response, 'error', None)),
            'incomplete_details': repr(getattr(response, 'incomplete_details', None)),
            'usage': repr(getattr(response, 'usage', None)),
            'retrieval_performed': True,
        }
    finally:
        client.close()
        credential.close()

if 'INVOCATION_RESULT' not in globals():
    raise RuntimeError('raw response diagnostics are blocked: run the approved invocation cell first so a recorded response_id exists')
RAW_RESPONSE_DIAGNOSTICS = retrieve_recorded_response(INVOCATION_RESULT)
RAW_RESPONSE_DIAGNOSTICS


{'response_id': 'resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99', 'status': 'completed', 'output_text': '{\n  "analyst_contract_version": 1,\n  "packet_sha256": "bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3",\n  "assessment": "This descriptive-smoke-only brief reports mean paired delta 0 across 2 matches for candidates 10, 40, and 60, and mean paired delta 937 across 2 matches for candidate 80. Human interpretation is required; the brief does not rank candidates or propose a new experiment.",\n  "citations": [\n    {\n      "packet_path": "aggregates[0]",\n      "claim": "candidate 10: mean paired delta 0 across 2 matches"\n    },\n    {\n      "packet_path": "aggregates[1]",\n      "claim": "candidate 40: mean paired delta 0 across 2 matches"\n    },\n    {\n      "packet_path": "aggregates[2]",\n      "claim": "candidate 60: mean paired delta 0 across 2 matches"\n    },\n    {\n      "packet_path": "aggregates[3]",\n      "claim": "candidate 80: mean pair

### 7.10a Pretty-print raw response diagnostics

This presentation cell formats only the retrieved response metadata. It does not call the service or infer quality from an empty response.

In [38]:
from IPython.display import Markdown, display

if 'RAW_RESPONSE_DIAGNOSTICS' not in globals():
    raise RuntimeError('raw response presentation is blocked: run 7.10 first and fix its reported error before rerunning this cell')
raw = RAW_RESPONSE_DIAGNOSTICS
display(Markdown(f"""### Raw response diagnostics

| Fact | Value |
|:---|:---|
| Response ID | `{raw.get('response_id', 'not available')}` |
| Status | `{raw.get('status', 'not available')}` |
| Output characters | `{raw.get('output_character_count', 'not available')}` |
| Response error | `{raw.get('error', 'not available')}` |
| Incomplete details | `{raw.get('incomplete_details', 'not available')}` |
| Retrieval performed | **{raw.get('retrieval_performed', False)}** |

The output text remains redacted from this diagnostic card; the character count and response metadata are the evidence used to choose the next debugging step.
"""))


### Raw response diagnostics

| Fact | Value |
|:---|:---|
| Response ID | `resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99` |
| Status | `completed` |
| Output characters | `1184` |
| Response error | `None` |
| Incomplete details | `None` |
| Retrieval performed | **True** |

The output text remains redacted from this diagnostic card; the character count and response metadata are the evidence used to choose the next debugging step.


### 7.10b Create or reuse a disabled Agent Insights monitor

The trace query is now wired to the installed beta SDK, but the project has no monitor. This cell creates exactly one disabled monitor for `evidence-analyst` when absent, or reuses an exact matching monitor. Disabled means no scheduled insight-generation run is started; the cell only establishes the readback surface.

In [39]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import AgentInsightMonitorCreate
from azure.identity import DefaultAzureCredential


def ensure_trace_monitor() -> dict[str, object]:
    """Create or reuse one exact disabled Agent Insights monitor."""
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
        allow_preview=True,
    )
    try:
        monitors_api = client.beta.agent_insight_monitors
        existing = tuple(monitors_api.list(agent_name='evidence-analyst'))
        if len(existing) > 1:
            raise RuntimeError('trace monitor setup found multiple monitors for evidence-analyst; stop and resolve the duplicate state before continuing')
        if existing:
            monitor_id = getattr(existing[0], 'id', None)
            monitor = monitors_api.get(monitor_id) if monitor_id else None
            if monitor is None:
                raise RuntimeError('trace monitor setup could not read back the existing monitor ID')
            if getattr(monitor, 'agent_name', None) != 'evidence-analyst' or getattr(monitor, 'model_deployment_name', None) != 'evidence-analyst-luna-v1' or getattr(monitor, 'enabled', None) is not False:
                raise ValueError('existing trace monitor differs from the reviewed disabled configuration; refusing to update it')
            action = 'reused_existing'
        else:
            monitor = monitors_api.create(
                AgentInsightMonitorCreate(
                    agent_name='evidence-analyst',
                    model_deployment_name='evidence-analyst-luna-v1',
                    enabled=False,
                    run_interval_hours=6,
                )
            )
            action = 'created'
        monitor_id = getattr(monitor, 'id', None)
        if not monitor_id:
            raise RuntimeError('trace monitor readback has no monitor ID')
        verified = monitors_api.get(monitor_id)
        return {
            'action': action,
            'monitor_id': getattr(verified, 'id', monitor_id),
            'agent_name': getattr(verified, 'agent_name', None),
            'model_deployment_name': getattr(verified, 'model_deployment_name', None),
            'enabled': getattr(verified, 'enabled', None),
            'run_interval_hours': getattr(verified, 'run_interval_hours', None),
        }
    finally:
        client.close()
        credential.close()

TRACE_MONITOR_READBACK = ensure_trace_monitor()
TRACE_MONITOR_READBACK


{'action': 'reused_existing', 'monitor_id': 'monitor_1cc62eccddcf46ed826fa08ed497a92f', 'agent_name': 'evidence-analyst', 'model_deployment_name': 'evidence-analyst-luna-v1', 'enabled': False, 'run_interval_hours': 6.0}

### 7.10c Pretty-print monitor readback

This card reports the actual monitor configuration and makes the disabled scheduling boundary visible. It does not trigger insight generation.

In [40]:
from IPython.display import Markdown, display

if 'TRACE_MONITOR_READBACK' not in globals():
    raise RuntimeError('trace monitor presentation is blocked: run 7.10b first and fix its reported error before rerunning this cell')
monitor = TRACE_MONITOR_READBACK
display(Markdown(f"""### Agent Insights monitor

| Fact | Value |
|:---|:---|
| Action | `{monitor.get('action', 'not available')}` |
| Monitor ID | `{monitor.get('monitor_id', 'not available')}` |
| Agent | `{monitor.get('agent_name', 'not available')}` |
| Analysis model deployment | `{monitor.get('model_deployment_name', 'not available')}` |
| Scheduled generation enabled | **{monitor.get('enabled', 'not available')}** |
| Run interval hours | `{monitor.get('run_interval_hours', 'not available')}` |

> No insight-generation run was started by this cell.
"""))


### Agent Insights monitor

| Fact | Value |
|:---|:---|
| Action | `reused_existing` |
| Monitor ID | `monitor_1cc62eccddcf46ed826fa08ed497a92f` |
| Agent | `evidence-analyst` |
| Analysis model deployment | `evidence-analyst-luna-v1` |
| Scheduled generation enabled | **False** |
| Run interval hours | `6.0` |

> No insight-generation run was started by this cell.


### 7.10f Create or reuse the Application Insights project connection

The project has an Application Insights resource but its named project connection was not marked as the default App Insights connection. This notebook cell discovers the existing resource in the approved resource group, then creates or repairs the same named project connection with the supported App Insights `ApiKey` shape and `isDefault=true`. The connection string is retrieved only in memory, never stored or displayed, and target/authentication drift is refused rather than duplicated.

In [41]:
from azure.identity import DefaultAzureCredential
import json
import urllib.error
import urllib.parse
import urllib.request


APPINSIGHTS_CONNECTION_NAME = 'foundry-trainer-appinsights'


def _management_json_request(method: str, url: str, credential, body: dict[str, object] | None = None):
    token = credential.get_token('https://management.azure.com/.default').token
    payload = json.dumps(body).encode('utf-8') if body is not None else None
    request = urllib.request.Request(
        url,
        data=payload,
        method=method,
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(request) as response:
            raw = response.read().decode('utf-8')
            return response.status, json.loads(raw) if raw else None
    except urllib.error.HTTPError as error:
        if error.code == 404:
            return 404, None
        raise


def ensure_appinsights_project_connection() -> dict[str, object]:
    """Discover and create/reuse one exact AAD project connection."""
    config = VALIDATED_DEPLOYMENT_CONFIG
    credential = DefaultAzureCredential()
    try:
        resource_query = urllib.parse.quote("resourceType eq 'Microsoft.Insights/components'")
        resources_url = (
            f'https://management.azure.com/subscriptions/{config.subscription_id}'
            f'/resourceGroups/{config.resource_group}/resources?api-version=2021-04-01'
            f'&$filter={resource_query}'
        )
        resource_status, resource_body = _management_json_request('GET', resources_url, credential)
        resources = tuple(resource_body.get('value', [])) if resource_body else ()
        if len(resources) != 1:
            raise RuntimeError(f'Application Insights discovery expected exactly one component in {config.resource_group}, found {len(resources)}')
        appinsights_id = resources[0].get('id')
        if not isinstance(appinsights_id, str) or not appinsights_id:
            raise RuntimeError('Application Insights discovery returned no resource ID')
        appinsights_url = f'https://management.azure.com{appinsights_id}?api-version=2020-02-02'
        appinsights_status, appinsights_body = _management_json_request('GET', appinsights_url, credential)
        appinsights_properties = appinsights_body.get('properties', {}) if appinsights_body else {}
        workspace_resource_id = (
            appinsights_properties.get('WorkspaceResourceId')
            or appinsights_properties.get('workspaceResourceId')
        )
        if not isinstance(workspace_resource_id, str) or not workspace_resource_id:
            raise RuntimeError(
                'Application Insights readback has no linked Log Analytics workspace; '
                'Agent Insights log-reader roles cannot be scoped safely'
            )
        appinsights_connection_string = appinsights_properties.get('ConnectionString') or appinsights_properties.get('connectionString')
        if not isinstance(appinsights_connection_string, str) or not appinsights_connection_string:
            raise RuntimeError('Application Insights resource has no connection string available to configure the Foundry project connection')
        connection_url = (
            f'https://management.azure.com/subscriptions/{config.subscription_id}'
            f'/resourceGroups/{config.resource_group}/providers/Microsoft.CognitiveServices'
            f'/accounts/{config.account_name}/projects/{config.project_name}/connections/{APPINSIGHTS_CONNECTION_NAME}'
            '?api-version=2025-06-01'
        )
        expected = {
            'category': 'AppInsights',
            'target': appinsights_id,
            'authType': 'ApiKey',
            'isDefault': True,
        }
        metadata = {
            'ApiType': 'Azure',
            'ResourceId': appinsights_id,
            'ApplicationInsightsConnectionString': appinsights_connection_string,
        }
        status, current = _management_json_request('GET', connection_url, credential)
        if status == 404:
            _, created = _management_json_request('PUT', connection_url, credential, {'properties': {**expected, 'credentials': {'key': appinsights_connection_string}, 'metadata': metadata}})
            action = 'created'
        else:
            current_properties = current.get('properties', {}) if current else {}
            for field in ('category', 'target', 'authType'):
                if current_properties.get(field) != expected[field]:
                    raise ValueError(f'existing Application Insights connection differs in {field}; refusing to overwrite it')
            if current_properties.get('isDefault') is not True:
                _, created = _management_json_request('PUT', connection_url, credential, {'properties': {**expected, 'credentials': {'key': appinsights_connection_string}, 'metadata': metadata}})
                action = 'updated_default'
            else:
                created = current
                action = 'reused_existing'
        verify_status, verified = _management_json_request('GET', connection_url, credential)
        if verify_status != 200 or not verified:
            raise RuntimeError('Application Insights project connection readback did not return HTTP 200')
        verified_properties = verified.get('properties', {})
        if any(verified_properties.get(field) != value for field, value in expected.items()):
            raise RuntimeError('Application Insights project connection readback differs from the requested target')
        return {
            'action': action,
            'connection_name': APPINSIGHTS_CONNECTION_NAME,
            'connection_id': verified.get('id'),
            'appinsights_resource_id': appinsights_id,
            'workspace_resource_id': workspace_resource_id,
            'category': verified_properties.get('category'),
            'auth_type': verified_properties.get('authType'),
            'is_default': verified_properties.get('isDefault'),
            'status': 'available',
        }
    finally:
        credential.close()

if 'VALIDATED_DEPLOYMENT_CONFIG' not in globals():
    raise RuntimeError('Application Insights connection setup requires Section 1 to be run in this kernel')
APPINSIGHTS_CONNECTION_RESULT = ensure_appinsights_project_connection()
APPINSIGHTS_CONNECTION_RESULT


{'action': 'reused_existing', 'connection_name': 'foundry-trainer-appinsights', 'connection_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/projects/foundry-trainer/connections/foundry-trainer-appinsights', 'appinsights_resource_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights', 'workspace_resource_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs', 'category': 'AppInsights', 'auth_type': 'ApiKey', 'is_default': True, 'status': 'available'}

### 7.10f Pretty-print the Application Insights connection readback

This card reports the actual connection identity and authentication mode without displaying connection strings or secrets.

In [42]:
from IPython.display import Markdown, display

if 'APPINSIGHTS_CONNECTION_RESULT' not in globals():
    raise RuntimeError('Application Insights connection presentation is blocked: run 7.10f first and fix its reported error')
connection = APPINSIGHTS_CONNECTION_RESULT
display(Markdown(f"""### Application Insights project connection

| Fact | Value |
|:---|:---|
| Action | `{connection.get('action', 'not available')}` |
| Connection name | `{connection.get('connection_name', 'not available')}` |
| Connection ID | `{connection.get('connection_id', 'not available')}` |
| Category | `{connection.get('category', 'not available')}` |
| Authentication | `{connection.get('auth_type', 'not available')}` |
| Status | **{connection.get('status', 'unknown')}** |

> No connection string or secret is displayed.
"""))


### Application Insights project connection

| Fact | Value |
|:---|:---|
| Action | `reused_existing` |
| Connection name | `foundry-trainer-appinsights` |
| Connection ID | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/projects/foundry-trainer/connections/foundry-trainer-appinsights` |
| Category | `AppInsights` |
| Authentication | `ApiKey` |
| Status | **available** |

> No connection string or secret is displayed.


### 7.10f Create or reuse the account-level shared App Insights connection

⚠️ **Assumption:** The Claims sample links Application Insights at the Foundry account level with `isSharedToAll=true`. The existing account-level shared connection is already present and remains the canonical shared resource; the earlier project-specific connection is kept as the project action surface and will be promoted to the default App Insights connection if needed. The existing connection string is used only in memory.

In [43]:
ACCOUNT_APPINSIGHTS_CONNECTION_NAME = 'foundry-trainer-appinsights-shared'

def ensure_account_appinsights_connection() -> dict[str, object]:
    config = VALIDATED_DEPLOYMENT_CONFIG
    credential = DefaultAzureCredential()
    try:
        resource_id = APPINSIGHTS_CONNECTION_RESULT['appinsights_resource_id']
        appinsights_url = f'https://management.azure.com{resource_id}?api-version=2020-02-02'
        token = credential.get_token('https://management.azure.com/.default').token
        request = urllib.request.Request(appinsights_url, headers={'Authorization': f'Bearer {token}'})
        with urllib.request.urlopen(request) as response:
            properties = json.loads(response.read().decode('utf-8')).get('properties', {})
        connection_string = properties.get('ConnectionString') or properties.get('connectionString')
        if not connection_string:
            raise RuntimeError('❌ account-level App Insights connection requires a readable connection string')
        account_connection_url = (
            f'https://management.azure.com/subscriptions/{config.subscription_id}/resourceGroups/{config.resource_group}'
            f'/providers/Microsoft.CognitiveServices/accounts/{config.account_name}/connections/{ACCOUNT_APPINSIGHTS_CONNECTION_NAME}'
            '?api-version=2025-06-01'
        )
        expected = {
            'category': 'AppInsights',
            'target': resource_id,
            'authType': 'ApiKey',
            'isSharedToAll': True,
            'metadata': {'ApiType': 'Azure', 'ResourceId': resource_id},
        }
        status, current = _management_json_request('GET', account_connection_url, credential)
        current_properties = current.get('properties', {}) if current else {}
        if status == 404:
            _management_json_request('PUT', account_connection_url, credential, {'properties': {**expected, 'credentials': {'key': connection_string}}})
            action = 'created'
        else:
            for field in ('category', 'target', 'authType', 'isSharedToAll'):
                if current_properties.get(field) != expected[field]:
                    raise ValueError(f'existing account-level App Insights connection differs in {field}; refusing overwrite')
            action = 'reused_existing'
        verify_status, verified = _management_json_request('GET', account_connection_url, credential)
        if verify_status != 200 or not verified:
            raise RuntimeError('❌ account-level App Insights connection readback failed')
        verified_properties = verified.get('properties', {})
        for field, value in expected.items():
            if verified_properties.get(field) != value:
                raise RuntimeError(f'account-level App Insights connection readback differs in {field}')
        return {
            'action': action,
            'connection_name': ACCOUNT_APPINSIGHTS_CONNECTION_NAME,
            'connection_id': verified.get('id'),
            'target': resource_id,
            'category': verified_properties.get('category'),
            'auth_type': verified_properties.get('authType'),
            'is_shared_to_all': verified_properties.get('isSharedToAll'),
            'metadata_keys': tuple(sorted((verified_properties.get('metadata') or {}).keys())),
            'status': 'available',
        }
    finally:
        credential.close()

if 'VALIDATED_DEPLOYMENT_CONFIG' not in globals() or 'APPINSIGHTS_CONNECTION_RESULT' not in globals():
    raise RuntimeError('❌ account-level App Insights connection setup requires Sections 1 and 7.10f')
ACCOUNT_APPINSIGHTS_CONNECTION_RESULT = ensure_account_appinsights_connection()
ACCOUNT_APPINSIGHTS_CONNECTION_RESULT


{'action': 'reused_existing', 'connection_name': 'foundry-trainer-appinsights-shared', 'connection_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/connections/foundry-trainer-appinsights-shared', 'target': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights', 'category': 'AppInsights', 'auth_type': 'ApiKey', 'is_shared_to_all': True, 'metadata_keys': ('ApiType', 'ResourceId'), 'status': 'available'}

### 7.10f Pretty-print the account-level connection readback

This card reports the account-level shared connection without displaying credentials.

In [44]:
from IPython.display import Markdown, display

if 'ACCOUNT_APPINSIGHTS_CONNECTION_RESULT' not in globals():
    raise RuntimeError('❌ account-level App Insights connection presentation is blocked: run its setup cell first')
connection = ACCOUNT_APPINSIGHTS_CONNECTION_RESULT
display(Markdown(f"""### Account-level App Insights connection

| Fact | Value |
|:---|:---|
| Action | `{connection.get('action', 'not available')}` |
| Connection name | `{connection.get('connection_name', 'not available')}` |
| Connection ID | `{connection.get('connection_id', 'not available')}` |
| Shared to all projects | **True** |
| Status | **{connection.get('status', 'unknown')}** |

> No connection string or secret is displayed.
"""))


### Account-level App Insights connection

| Fact | Value |
|:---|:---|
| Action | `reused_existing` |
| Connection name | `foundry-trainer-appinsights-shared` |
| Connection ID | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/connections/foundry-trainer-appinsights-shared` |
| Shared to all projects | **True** |
| Status | **available** |

> No connection string or secret is displayed.


### 7.10f Assign or reuse trace prerequisites

The project connection exists, but Foundry rejected the insight run because a required telemetry dependency permission was missing. Microsoft monitoring guidance distinguishes the Application Insights component from its linked Log Analytics workspace: write/publish access stays on Application Insights, while log readers belong on the workspace. This cell discovers that linked workspace from the already-read resource, then idempotently assigns only the exact roles needed by the project identity and signed-in caller. It reads each assignment first, reuses exact matches, and refuses drift.

⚠️ **Assumption:** Agent Insights may read protected `AppGenAIContent` data while analyzing prompt/response traces. The `Privileged Monitoring Data Reader` role is therefore added only at the linked workspace scope for the project identity and caller. This is a sensitive-data access decision, not a general subscription grant, and remains visible for later review.

In [45]:
from azure.ai.projects import AIProjectClient
import base64
import json
import uuid


FOUNDRY_USER_ROLE_ID = '53ca6127-db72-4b80-b1b0-d745d6d5456d'
MONITORING_METRICS_PUBLISHER_ROLE_ID = '3913510d-42f4-4e42-8a64-420c390055eb'
LOG_ANALYTICS_READER_ROLE_ID = '73c42c96-874c-492b-b04d-ab87d138a893'
MONITORING_READER_ROLE_ID = '43d0d8ad-25c7-4714-9337-8ba259a9fe05'


def resolve_builtin_role_id(role_name: str, credential) -> str:
    """Resolve a built-in role by its immutable display name before assignment."""
    encoded_name = urllib.parse.quote(role_name, safe="")
    role_url = (
        'https://management.azure.com/providers/Microsoft.Authorization/roleDefinitions'
        f'?api-version=2022-04-01&$filter=roleName%20eq%20%20%27{encoded_name}%27'
    )
    status, body = _management_json_request('GET', role_url, credential)
    values = tuple((body or {}).get('value', ()))
    if status != 200 or len(values) != 1:
        raise RuntimeError(
            f'trace role setup could not resolve exactly one built-in role named {role_name!r}'
        )
    role_id = values[0].get('name') or str(values[0].get('id', '')).rsplit('/', 1)[-1]
    if not role_id:
        raise RuntimeError(f'trace role setup resolved {role_name!r} without a role ID')
    return role_id


def ensure_trace_role_assignments() -> dict[str, object]:
    """Assign only the two reviewed trace prerequisites, reusing exact matches."""
    if 'APPINSIGHTS_CONNECTION_RESULT' not in globals() or APPINSIGHTS_CONNECTION_RESULT.get('status') != 'available':
        raise RuntimeError('trace role setup is blocked: create and read back the Application Insights project connection first')
    config = VALIDATED_DEPLOYMENT_CONFIG
    account_id = (
        f'/subscriptions/{config.subscription_id}/resourceGroups/{config.resource_group}'
        f'/providers/Microsoft.CognitiveServices/accounts/{config.account_name}'
    )
    project_id = f'{account_id}/projects/{config.project_name}'
    appinsights_id = APPINSIGHTS_CONNECTION_RESULT['appinsights_resource_id']
    workspace_id = APPINSIGHTS_CONNECTION_RESULT.get('workspace_resource_id')
    if not isinstance(workspace_id, str) or not workspace_id:
        raise RuntimeError('trace role setup requires the linked Log Analytics workspace resource ID from the connection readback')
    credential = DefaultAzureCredential()
    try:
        account_url = f'https://management.azure.com{account_id}?api-version=2025-06-01'
        account_status, account_body = _management_json_request('GET', account_url, credential)
        project_url = f'https://management.azure.com{project_id}?api-version=2025-06-01'
        project_status, project_body = _management_json_request('GET', project_url, credential)
        account_principal_id = (account_body or {}).get('identity', {}).get('principalId')
        project_principal_id = (project_body or {}).get('identity', {}).get('principalId')
        if not isinstance(project_principal_id, str) or not project_principal_id:
            raise RuntimeError('trace role setup could not find the Foundry project child resource managed identity principal ID')
        management_token = credential.get_token('https://management.azure.com/.default').token
        token_payload = management_token.split('.')[1] + '==='
        caller_principal_id = json.loads(base64.urlsafe_b64decode(token_payload.encode('ascii'))).get('oid')
        if not isinstance(caller_principal_id, str) or not caller_principal_id:
            raise RuntimeError('trace role setup could not identify the signed-in caller object ID for Log Analytics Reader')
        project_client = AIProjectClient(endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer', credential=credential)
        try:
            agent_details = project_client.agents.get('evidence-analyst')
            agent_identity = getattr(agent_details, 'instance_identity', None)
            agent_principal_id = getattr(agent_identity, 'principal_id', None)
        finally:
            project_client.close()
        if not isinstance(agent_principal_id, str) or not agent_principal_id:
            raise RuntimeError('trace role setup could not find the evidence-analyst instance identity principal ID')
        privileged_monitoring_role_id = resolve_builtin_role_id('Privileged Monitoring Data Reader', credential)
        assignments = (
            (account_id, FOUNDRY_USER_ROLE_ID, 'Foundry User (account identity)', account_principal_id, 'ServicePrincipal'),
            (account_id, FOUNDRY_USER_ROLE_ID, 'Foundry User (caller)', caller_principal_id, 'User'),
            (project_id, FOUNDRY_USER_ROLE_ID, 'Foundry User (project identity)', project_principal_id, 'ServicePrincipal'),
            (project_id, FOUNDRY_USER_ROLE_ID, 'Foundry User (caller at project scope)', caller_principal_id, 'User'),
            (appinsights_id, MONITORING_METRICS_PUBLISHER_ROLE_ID, 'Monitoring Metrics Publisher (account identity)', account_principal_id, 'ServicePrincipal'),
            (appinsights_id, MONITORING_METRICS_PUBLISHER_ROLE_ID, 'Monitoring Metrics Publisher (project identity)', project_principal_id, 'ServicePrincipal'),
            (appinsights_id, privileged_monitoring_role_id, 'Privileged Monitoring Data Reader (project identity at App Insights)', project_principal_id, 'ServicePrincipal'),
            (workspace_id, LOG_ANALYTICS_READER_ROLE_ID, 'Log Analytics Reader', caller_principal_id, 'User'),
            (workspace_id, MONITORING_READER_ROLE_ID, 'Monitoring Reader', caller_principal_id, 'User'),
            (workspace_id, privileged_monitoring_role_id, 'Privileged Monitoring Data Reader', caller_principal_id, 'User'),
            (workspace_id, LOG_ANALYTICS_READER_ROLE_ID, 'Log Analytics Reader (project identity)', project_principal_id, 'ServicePrincipal'),
            (workspace_id, MONITORING_READER_ROLE_ID, 'Monitoring Reader (project identity)', project_principal_id, 'ServicePrincipal'),
            (workspace_id, privileged_monitoring_role_id, 'Privileged Monitoring Data Reader (project identity)', project_principal_id, 'ServicePrincipal'),
            (workspace_id, LOG_ANALYTICS_READER_ROLE_ID, 'Log Analytics Reader (account identity)', account_principal_id, 'ServicePrincipal'),
            (workspace_id, MONITORING_READER_ROLE_ID, 'Monitoring Reader (account identity)', account_principal_id, 'ServicePrincipal'),
            (workspace_id, privileged_monitoring_role_id, 'Privileged Monitoring Data Reader (account identity)', account_principal_id, 'ServicePrincipal'),
            (appinsights_id, MONITORING_METRICS_PUBLISHER_ROLE_ID, 'Monitoring Metrics Publisher (agent identity)', agent_principal_id, 'ServicePrincipal'),
            (workspace_id, LOG_ANALYTICS_READER_ROLE_ID, 'Log Analytics Reader (agent identity)', agent_principal_id, 'ServicePrincipal'),
            (workspace_id, MONITORING_READER_ROLE_ID, 'Monitoring Reader (agent identity)', agent_principal_id, 'ServicePrincipal'),
            (workspace_id, privileged_monitoring_role_id, 'Privileged Monitoring Data Reader (agent identity)', agent_principal_id, 'ServicePrincipal'),
        )
        results = []
        for scope, role_id, role_name, assignee_id, assignee_type in assignments:
            list_url = f'https://management.azure.com{scope}/providers/Microsoft.Authorization/roleAssignments?api-version=2022-04-01&$filter=principalId%20eq%20%27{assignee_id}%27'
            _, current = _management_json_request('GET', list_url, credential)
            values = tuple((current or {}).get('value', ()))
            role_suffix = f'/providers/Microsoft.Authorization/roleDefinitions/{role_id}'.lower()
            matching = next((
                item for item in values
                if str(item.get('properties', {}).get('roleDefinitionId', '')).lower().endswith(role_suffix)
                and item.get('properties', {}).get('scope') == scope
            ), None)
            if matching is not None:
                action = 'reused_existing'
                assignment_id = matching.get('name')
            else:
                assignment_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{scope}|{assignee_id}|{role_id}'))
                assignment_url = f'https://management.azure.com{scope}/providers/Microsoft.Authorization/roleAssignments/{assignment_id}?api-version=2022-04-01'
                _management_json_request('PUT', assignment_url, credential, {
                    'properties': {
                        'roleDefinitionId': f'/subscriptions/{config.subscription_id}/providers/Microsoft.Authorization/roleDefinitions/{role_id}',
                        'principalId': assignee_id,
                        'principalType': assignee_type,
                    }
                })
                action = 'created'
            results.append({'role': role_name, 'scope': scope, 'action': action, 'assignment_id': assignment_id})
        return {'status': 'available', 'account_principal_id': account_principal_id, 'principal_id': project_principal_id, 'project_id': project_id, 'workspace_id': workspace_id, 'assignments': tuple(results)}
    finally:
        credential.close()

TRACE_ROLE_ASSIGNMENTS = ensure_trace_role_assignments()
TRACE_ROLE_ASSIGNMENTS


{'status': 'available', 'account_principal_id': '7bc9278e-19c9-492d-9794-44ce6c45815f', 'principal_id': 'f29cfdf5-5e0b-4aa7-98da-0266c9b72cae', 'project_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/projects/foundry-trainer', 'workspace_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs', 'assignments': ({'role': 'Foundry User (account identity)', 'scope': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource', 'action': 'reused_existing', 'assignment_id': '624a611b-1d2b-5232-807d-a1463141de08'}, {'role': 'Foundry User (caller)', 'scope': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer

### 7.10f Pretty-print trace prerequisite roles

This card reports role names, scopes, and reuse/create actions. It intentionally redacts the managed-identity principal ID.

In [46]:
from IPython.display import Markdown, display

if 'TRACE_ROLE_ASSIGNMENTS' not in globals():
    raise RuntimeError('trace role presentation is blocked: run 7.10f role setup first')
roles = TRACE_ROLE_ASSIGNMENTS
rows = '\n'.join(f"| {item['role']} | `{item['action']}` | `{item['scope']}` |" for item in roles.get('assignments', ()))
display(Markdown(f"""### Trace prerequisite roles

| Role | Action | Scope |
|:---|:---|:---|
{rows}

> The managed-identity principal ID is intentionally redacted.
"""))


### Trace prerequisite roles

| Role | Action | Scope |
|:---|:---|:---|
| Foundry User (account identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource` |
| Foundry User (caller) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource` |
| Foundry User (project identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/projects/foundry-trainer` |
| Foundry User (caller at project scope) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.CognitiveServices/accounts/foundry-trainer-resource/projects/foundry-trainer` |
| Monitoring Metrics Publisher (account identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights` |
| Monitoring Metrics Publisher (project identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights` |
| Privileged Monitoring Data Reader (project identity at App Insights) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights` |
| Log Analytics Reader | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Monitoring Reader | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Privileged Monitoring Data Reader | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Log Analytics Reader (project identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Monitoring Reader (project identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Privileged Monitoring Data Reader (project identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Log Analytics Reader (account identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Monitoring Reader (account identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Privileged Monitoring Data Reader (account identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Monitoring Metrics Publisher (agent identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights` |
| Log Analytics Reader (agent identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Monitoring Reader (agent identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |
| Privileged Monitoring Data Reader (agent identity) | `reused_existing` | `/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.OperationalInsights/workspaces/foundry-trainer-resource-logs` |

> The managed-identity principal ID is intentionally redacted.


### 7.10g Enforce Entra-only Application Insights authentication

⚠️ **Assumption:** The supported project connection is ApiKey-backed with the existing App Insights connection string held only in memory. This cell reads the Application Insights resource's local-auth setting and enables local authentication when needed for that connection. It never displays the connection string; reruns reuse the already-correct setting.

In [47]:
def ensure_appinsights_entra_auth() -> dict[str, object]:
    resource_id = APPINSIGHTS_CONNECTION_RESULT['appinsights_resource_id']
    credential = DefaultAzureCredential()
    try:
        resource_url = f'https://management.azure.com{resource_id}?api-version=2020-02-02'
        status, current = _management_json_request('GET', resource_url, credential)
        properties = current.get('properties', {}) if current else {}
        if properties.get('DisableLocalAuth') is False:
            action = 'reused_existing'
        else:
            patch_status, _ = _management_json_request('PATCH', resource_url, credential, {'properties': {'DisableLocalAuth': False}})
            if patch_status not in (200, 202):
                raise RuntimeError(f'Application Insights Entra-auth update returned HTTP {patch_status}')
            action = 'updated'
        verify_status, verified = _management_json_request('GET', resource_url, credential)
        verified_value = (verified or {}).get('properties', {}).get('DisableLocalAuth')
        if verified_value is not False:
            raise RuntimeError('Application Insights readback did not confirm local authentication is enabled for the ApiKey connection')
        return {'action': action, 'resource_id': resource_id, 'disable_local_auth': verified_value, 'status': 'available'}
    finally:
        credential.close()

if 'APPINSIGHTS_CONNECTION_RESULT' not in globals():
    raise RuntimeError('Application Insights authentication setup is blocked: create the project connection first')
APPINSIGHTS_AUTH_RESULT = ensure_appinsights_entra_auth()
APPINSIGHTS_AUTH_RESULT


{'action': 'reused_existing', 'resource_id': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights', 'disable_local_auth': False, 'status': 'available'}

### 7.10g Pretty-print Application Insights authentication readback

This card reports only the authentication mode and update/reuse action.

In [48]:
from IPython.display import Markdown, display

if 'APPINSIGHTS_AUTH_RESULT' not in globals():
    raise RuntimeError('Application Insights authentication presentation is blocked: run 7.10g first')
auth = APPINSIGHTS_AUTH_RESULT
display(Markdown(f"""### Application Insights authentication

| Fact | Value |
|:---|:---|
| Action | `{auth.get('action', 'not available')}` |
| Local authentication disabled | **{auth.get('disable_local_auth', 'not available')}** |
| Status | **{auth.get('status', 'unknown')}** |
"""))


### Application Insights authentication

| Fact | Value |
|:---|:---|
| Action | `reused_existing` |
| Local authentication disabled | **False** |
| Status | **available** |


### 7.10d Run one bounded Agent Insights analysis

This cell starts one on-demand analysis over the previous 168 hours of retained traces. It does not invoke `evidence-analyst`, enable scheduled monitoring, or change the deployment. It may consume tokens on the analysis model, so the exact lookback and deterministic operation ID are recorded.

⚠️ **Assumption:** Application Insights connection, authentication mode, and required role assignments are already read back as available. If not, this cell records a blocker and stops.

In [49]:
from azure.ai.projects.models import AgentInsightRunCreate
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


def run_bounded_trace_analysis(monitor: dict[str, object]) -> dict[str, object]:
    """Run one bounded on-demand insights analysis and return safe statistics."""
    if 'APPINSIGHTS_CONNECTION_RESULT' not in globals() or APPINSIGHTS_CONNECTION_RESULT.get('status') != 'available':
        raise RuntimeError('trace analysis is blocked: create and read back the Application Insights project connection before starting this run')
    monitor_id = monitor.get('monitor_id')
    if not monitor_id:
        raise RuntimeError('trace analysis is blocked: the monitor readback has no monitor_id; run monitor setup first')
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
        allow_preview=True,
    )
    try:
        operation_id = 'carrot-evidence-analyst-insight-run-v17'
        poller = client.beta.agent_insight_monitors.begin_create_run(
            monitor_id,
            AgentInsightRunCreate(lookback_hours=168),
            operation_id=operation_id,
        )
        result = poller.result()
        details = getattr(poller, 'details', {}) or {}
        return {
            'operation_id': operation_id,
            'monitor_id': monitor_id,
            'lookback_hours': 168,
            'run_id': details.get('run_id') if isinstance(details, dict) else None,
            'traces_in_window': getattr(result, 'traces_in_window', None),
            'traces_analyzed': getattr(result, 'traces_analyzed', None),
            'insights_created': getattr(result, 'insights_created', None),
            'insights_updated': getattr(result, 'insights_updated', None),
            'insights_reopened': getattr(result, 'insights_reopened', None),
            'token_usage': repr(getattr(result, 'token_usage', None)),
        }
    finally:
        client.close()
        credential.close()

if 'TRACE_MONITOR_READBACK' not in globals():
    raise RuntimeError('trace analysis is blocked: run the disabled monitor setup cell before starting an analysis')
try:
    TRACE_RUN_RESULT = run_bounded_trace_analysis(TRACE_MONITOR_READBACK)
except Exception as error:
    TRACE_RUN_RESULT = {'status': 'blocked', 'error': str(error), 'operation_id': 'carrot-evidence-analyst-insight-run-v17'}
    raise
TRACE_RUN_RESULT


HttpResponseError: (Forbidden) Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.
Code: Forbidden
Message: Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.

### 7.10e Pretty-print the bounded trace analysis

This card shows the actual run statistics and token-usage record. It does not interpret the insights or turn them into a policy recommendation.

In [50]:
from IPython.display import Markdown, display

if 'TRACE_RUN_RESULT' not in globals():
    raise RuntimeError('trace analysis presentation is blocked: run 7.10d first and fix its reported error before rerunning this cell')
run = TRACE_RUN_RESULT
if run.get('status') == 'blocked' or 'FAILED' in str(run.get('status', '')).upper():
    display(Markdown(f"""# ❌ Agent Insights run blocked

{run.get('error', 'No blocker detail was recorded.')}
"""))
display(Markdown(f"""### Bounded Agent Insights run

| Fact | Value |
|:---|:---|
| Operation ID | `{run.get('operation_id', 'not available')}` |
| Monitor ID | `{run.get('monitor_id', 'not available')}` |
| Lookback hours | `{run.get('lookback_hours', 'not available')}` |
| Traces in window | `{run.get('traces_in_window', 'not available')}` |
| Traces analyzed | `{run.get('traces_analyzed', 'not available')}` |
| Insights created | `{run.get('insights_created', 'not available')}` |
| Insights updated | `{run.get('insights_updated', 'not available')}` |
| Insights reopened | `{run.get('insights_reopened', 'not available')}` |
| Token usage | `{run.get('token_usage', 'not available')}` |

> These are service statistics, not a recommendation or a policy decision.
"""))


# ❌ Agent Insights run blocked

(Forbidden) Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.
Code: Forbidden
Message: Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.


### Bounded Agent Insights run

| Fact | Value |
|:---|:---|
| Operation ID | `carrot-evidence-analyst-insight-run-v17` |
| Monitor ID | `not available` |
| Lookback hours | `not available` |
| Traces in window | `not available` |
| Traces analyzed | `not available` |
| Insights created | `not available` |
| Insights updated | `not available` |
| Insights reopened | `not available` |
| Token usage | `not available` |

> These are service statistics, not a recommendation or a policy decision.


### 7.10g Read back the bounded Agent Insights run

The start call returns a run ID, while the server may populate status and statistics asynchronously. This read-only cell retrieves that exact run and records its lifecycle, window, result counts, and any service error before the notebook interprets insights.

In [51]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def read_bounded_trace_run(run_result: dict[str, object]) -> dict[str, object]:
    """Read back one recorded Agent Insights run without starting another."""
    monitor_id = run_result.get('monitor_id')
    run_id = run_result.get('run_id')
    if not monitor_id or not run_id:
        raise RuntimeError('trace run readback is blocked: the bounded run result has no monitor_id or run_id')
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
        allow_preview=True,
    )
    try:
        run = client.beta.agent_insight_monitors.get_run(monitor_id, run_id)
        result = getattr(run, 'result', None)
        return {
            'monitor_id': monitor_id,
            'run_id': run_id,
            'status': str(getattr(run, 'status', None)),
            'agent_name': getattr(run, 'agent_name', None),
            'trigger': str(getattr(run, 'trigger', None)),
            'window_start': str(getattr(run, 'window_start', None)),
            'window_end': str(getattr(run, 'window_end', None)),
            'traces_in_window': getattr(result, 'traces_in_window', None) if result else None,
            'traces_analyzed': getattr(result, 'traces_analyzed', None) if result else None,
            'insights_created': getattr(result, 'insights_created', None) if result else None,
            'insights_updated': getattr(result, 'insights_updated', None) if result else None,
            'insights_reopened': getattr(result, 'insights_reopened', None) if result else None,
            'token_usage': repr(getattr(result, 'token_usage', None)) if result else None,
            'error': repr(getattr(run, 'error', None)),
        }
    finally:
        client.close()
        credential.close()

if 'TRACE_RUN_RESULT' not in globals():
    raise RuntimeError('❌ trace run readback requires the bounded run cell to execute first')
if TRACE_RUN_RESULT.get('status') == 'blocked':
    TRACE_RUN_READBACK = dict(TRACE_RUN_RESULT)
    raise RuntimeError(f"❌ trace run readback cannot proceed: {TRACE_RUN_RESULT.get('error', 'bounded run was blocked')}")
TRACE_RUN_READBACK = read_bounded_trace_run(TRACE_RUN_RESULT)
TRACE_RUN_READBACK


RuntimeError: ❌ trace run readback cannot proceed: (Forbidden) Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.
Code: Forbidden
Message: Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.

### 7.10h Pretty-print the Agent Insights run readback

This card reports the service-owned run status and statistics, including any error.

In [52]:
from IPython.display import Markdown, display

if 'TRACE_RUN_READBACK' not in globals():
    raise RuntimeError('trace run readback presentation is blocked: run 7.10g first')
run = TRACE_RUN_READBACK
if run.get('status') == 'blocked':
    display(Markdown(f"""# ❌ Agent Insights readback blocked

{run.get('error', 'No blocker detail was recorded.')}
"""))
display(Markdown(f"""### Agent Insights run readback

| Fact | Value |
|:---|:---|
| Run ID | `{run.get('run_id', 'not available')}` |
| Status | **{run.get('status', 'not available')}** |
| Trigger | `{run.get('trigger', 'not available')}` |
| Traces in window | `{run.get('traces_in_window', 'not available')}` |
| Traces analyzed | `{run.get('traces_analyzed', 'not available')}` |
| Insights created | `{run.get('insights_created', 'not available')}` |
| Insights updated | `{run.get('insights_updated', 'not available')}` |
| Token usage | `{run.get('token_usage', 'not available')}` |
| Service error | `{run.get('error', 'None')}` |
"""))


# ❌ Agent Insights readback blocked

(Forbidden) Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.
Code: Forbidden
Message: Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.


### Agent Insights run readback

| Fact | Value |
|:---|:---|
| Run ID | `not available` |
| Status | **blocked** |
| Trigger | `not available` |
| Traces in window | `not available` |
| Traces analyzed | `not available` |
| Insights created | `not available` |
| Insights updated | `not available` |
| Token usage | `not available` |
| Service error | `(Forbidden) Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.
Code: Forbidden
Message: Your identity does not have permission to access a required Agent Insights dependency. Verify your role assignments on the project connections, then retry.` |


### 7.11 Query the configured Foundry trace/insights surface

This is a read-only project query. It uses the installed SDK's beta `client.beta.agent_insight_monitors` surface to list Agent Insights monitors for `evidence-analyst`, then reads their available insights and searches the returned evidence for the recorded response ID. If no monitor exists, the cell raises a precise error explaining that trace retrieval has not been configured; it does not create one.

In [53]:
import json
from collections.abc import Mapping


def _jsonable_sdk_value(value):
    if hasattr(value, 'as_dict'):
        return value.as_dict()
    if isinstance(value, Mapping):
        return {str(key): _jsonable_sdk_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable_sdk_value(item) for item in value]
    return value


def query_recorded_trace_surface(invocation: dict[str, object]) -> dict[str, object]:
    """Read configured Agent Insights evidence without creating a monitor."""
    response_id = invocation.get('response_id')
    if not response_id:
        raise RuntimeError('trace query is blocked: the invocation has no response_id; run the approved invocation cell first')
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
        allow_preview=True,
    )
    try:
        beta_client = getattr(client, 'beta', None)
        monitors_api = getattr(beta_client, 'agent_insight_monitors', None)
        if monitors_api is None:
            raise RuntimeError('trace query is blocked: this installed azure-ai-projects client exposes no beta agent_insight_monitors operation; review the approved Foundry telemetry/insights SDK surface before changing the environment')
        monitors = list(monitors_api.list(agent_name=invocation.get('agent_name')))
        if not monitors:
            raise RuntimeError('trace query is blocked: no Agent Insights monitor exists for evidence-analyst; configure and approve a monitor in a separate notebook step before rerunning this read')
        monitor_records = []
        matching_insights = []
        for monitor in monitors:
            monitor_record = _jsonable_sdk_value(monitor)
            monitor_records.append(monitor_record)
            monitor_id = monitor_record.get('id') if isinstance(monitor_record, Mapping) else getattr(monitor, 'id', None)
            if not monitor_id:
                continue
            insights = list(monitors_api.list_insights(monitor_id, include_details=True))
            for insight in insights:
                record = _jsonable_sdk_value(insight)
                if response_id in json.dumps(record, sort_keys=True, default=str):
                    matching_insights.append(record)
        return {
            'response_id': response_id,
            'trace_query_performed': True,
            'monitor_count': len(monitor_records),
            'matching_insight_count': len(matching_insights),
            'matching_insights': matching_insights,
        }
    finally:
        client.close()
        credential.close()

if 'INVOCATION_RESULT' not in globals():
    raise RuntimeError('trace query is blocked: run the approved invocation cell first so a recorded response_id exists')
try:
    TRACE_QUERY_RESULT = query_recorded_trace_surface(INVOCATION_RESULT)
except (RuntimeError, AttributeError) as error:
    TRACE_QUERY_RESULT = {
        'response_id': INVOCATION_RESULT.get('response_id'),
        'trace_query_performed': False,
        'status': 'blocked',
        'error': str(error),
    }
    raise
TRACE_QUERY_RESULT


{'response_id': 'resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99', 'trace_query_performed': True, 'monitor_count': 1, 'matching_insight_count': 0, 'matching_insights': []}

### 7.11a Pretty-print trace query result

This card reports the actual read result or the exact actionable blocker captured by 7.11. It never hides a missing monitor or turns an empty result into a quality conclusion.

In [54]:
from IPython.display import Markdown, display

if 'TRACE_QUERY_RESULT' not in globals():
    raise RuntimeError('trace query presentation is blocked: run 7.11 first and fix its reported error before rerunning this cell')
trace = TRACE_QUERY_RESULT
display(Markdown(f"""### Trace query result

| Fact | Value |
|:---|:---|
| Response ID | `{trace.get('response_id', 'not available')}` |
| Query performed | **{trace.get('trace_query_performed', False)}** |
| Monitor count | `{trace.get('monitor_count', 'not available')}` |
| Matching insight count | `{trace.get('matching_insight_count', 'not available')}` |
| Status | `{trace.get('status', 'completed')}` |

**Blocker, if any**

{trace.get('error', 'No blocker was recorded.')}`

- [ ] Review the trace surface result before changing the invocation or enabling any new telemetry resource.
"""))


### Trace query result

| Fact | Value |
|:---|:---|
| Response ID | `resp_0017cf238c7e43e4006ab5a9f35ce881969284d851b5b6fa99` |
| Query performed | **True** |
| Monitor count | `1` |
| Matching insight count | `0` |
| Status | `completed` |

**Blocker, if any**

No blocker was recorded.`

- [ ] Review the trace surface result before changing the invocation or enabling any new telemetry resource.


### 7.10f Inspect the Application Insights project connection

The manual insight run failed before analysis because Foundry could not find a valid Application Insights connection. This read-only cell lists connection identity and type only; it never reads credentials or changes the project.

In [55]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType
from azure.identity import DefaultAzureCredential


def inspect_appinsights_connections() -> dict[str, object]:
    """List safe Application Insights connection facts without credentials."""
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
    )
    try:
        connections = tuple(client.connections.list(connection_type=ConnectionType.APPLICATION_INSIGHTS))
        records = tuple({
            'name': getattr(item, 'name', None),
            'connection_type': str(getattr(item, 'connection_type', None)),
            'target': getattr(item, 'target', None),
            'is_default': getattr(item, 'is_default', None),
        } for item in connections)
        if not records:
            raise RuntimeError('Application Insights connection is missing: create or attach one to this Foundry project before rerunning the bounded insight analysis')
        return {'status': 'available', 'connection_count': len(records), 'connections': records}
    finally:
        client.close()
        credential.close()

try:
    APPINSIGHTS_CONNECTION_RESULT = inspect_appinsights_connections()
except RuntimeError as error:
    APPINSIGHTS_CONNECTION_RESULT = {'status': 'blocked', 'error': str(error), 'connection_count': 0}
    raise
APPINSIGHTS_CONNECTION_RESULT


{'status': 'available', 'connection_count': 2, 'connections': ({'name': 'foundry-trainer-appinsights', 'connection_type': 'None', 'target': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights', 'is_default': True}, {'name': 'foundry-trainer-appinsights-shared', 'connection_type': 'None', 'target': '/subscriptions/d73a2af6-1ff7-42c1-91e7-a6bcc3d6f930/resourceGroups/rg-foundry-trainer-eastus2/providers/Microsoft.Insights/components/foundry-trainer-resource-appinsights', 'is_default': False})}

### 7.10g Pretty-print the Application Insights prerequisite

This card reports the actual safe connection inventory or the exact prerequisite error. It never displays credentials.

In [56]:
from IPython.display import Markdown, display

if 'APPINSIGHTS_CONNECTION_RESULT' not in globals():
    raise RuntimeError('Application Insights prerequisite presentation is blocked: run 7.10f first')
result = APPINSIGHTS_CONNECTION_RESULT
display(Markdown(f"""### Application Insights connection prerequisite

| Fact | Value |
|:---|:---|
| Status | **{result.get('status', 'unknown')}** |
| Connection count | `{result.get('connection_count', 'not available')}` |
| Error | `{result.get('error', 'None')}` |

> Connection credentials are intentionally not displayed.
"""))


### Application Insights connection prerequisite

| Fact | Value |
|:---|:---|
| Status | **available** |
| Connection count | `2` |
| Error | `None` |

> Connection credentials are intentionally not displayed.


## 7.12 Microsoft claims-sample tracing reproduction

This bounded reproduction follows the exact order in the Microsoft claims sample: tracing is configured before the SDK call, one temporary prompt-agent version receives one redacted request, and cleanup runs in `finally`. The temporary agent name is source-visible and unique to this experiment; an interrupted rerun reuses that exact name rather than creating a duplicate, then removes it after the call.

⚠️ **Assumption:** This makes one model call and may incur a small inference charge. It is not an Agent Insights run and it does not alter the durable `evidence-analyst` agent. The response is used only to test the sample tracing path.

The cleanup receipt is part of the evidence: a successful reproduction must show both the temporary agent and conversation were removed.

In [49]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

SAMPLE_REPRO_AGENT_NAME = 'carrot-sample-tracing-repro-v1'
SAMPLE_REPRO_INSTRUCTIONS = (
    'You are a tracing reproduction agent. Return a concise acknowledgement of the '
    'redacted test input. Do not claim a game result or make a recommendation.'
)
SAMPLE_REPRO_INPUT = 'Redacted tracing reproduction: report only that the sample path was reached.'

def run_claims_sample_tracing_reproduction() -> dict[str, object]:
    """Run the sample's one-call tracing path and always clean up its temporary resources."""
    if globals().get('TRACING_BOOTSTRAP_RESULT', {}).get('status') != 'available':
        raise RuntimeError('❌ sample tracing reproduction requires the tracing bootstrap to be available')
    credential = DefaultAzureCredential()
    client = AIProjectClient(
        endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
        credential=credential,
    )
    conversation_id = None
    agent_version = None
    cleanup_errors = []
    try:
        # Read first: an interrupted run may have left our named temporary agent.
        existing = tuple(client.agents.list())
        existing_name = next((item for item in existing if getattr(item, 'name', None) == SAMPLE_REPRO_AGENT_NAME), None)
        versions = tuple(client.agents.list_versions(SAMPLE_REPRO_AGENT_NAME, include_drafts=False)) if existing_name else ()
        reusable = next((item for item in versions if getattr(item, 'version', None)), None)
        if reusable is None:
            agent = client.agents.create_version(
                agent_name=SAMPLE_REPRO_AGENT_NAME,
                definition=PromptAgentDefinition(
                    model='evidence-analyst-luna-v1',
                    instructions=SAMPLE_REPRO_INSTRUCTIONS,
                    tools=[],
                ),
            )
            agent_action = 'created'
        else:
            agent = client.agents.get_version(SAMPLE_REPRO_AGENT_NAME, reusable.version)
            agent_action = 'reused_interrupted_run'
        agent_version = getattr(agent, 'version', None)
        if not agent_version:
            raise RuntimeError('sample tracing reproduction could not read the temporary agent version')
        openai_client = client.get_openai_client()
        conversation = openai_client.conversations.create()
        conversation_id = getattr(conversation, 'id', None)
        response = openai_client.responses.create(
            input=SAMPLE_REPRO_INPUT,
            conversation=conversation_id,
            extra_body={'agent_reference': {'name': SAMPLE_REPRO_AGENT_NAME, 'type': 'agent_reference'}},
        )
        result = {
            'status': 'response_received',
            'agent_name': SAMPLE_REPRO_AGENT_NAME,
            'agent_version': agent_version,
            'agent_action': agent_action,
            'response_id': getattr(response, 'id', None),
            'output_preview': getattr(response, 'output_text', '')[:240],
            'trace_bootstrap': 'available',
        }
    except Exception as error:
        result = {'status': 'blocked', 'error': str(error), 'agent_name': SAMPLE_REPRO_AGENT_NAME}
        raise
    finally:
        if conversation_id:
            try:
                client.get_openai_client().conversations.delete(conversation_id=conversation_id)
            except Exception as error:
                cleanup_errors.append(f'conversation cleanup: {error}')
        if agent_version:
            try:
                client.agents.delete_version(agent_name=SAMPLE_REPRO_AGENT_NAME, agent_version=agent_version)
            except Exception as error:
                cleanup_errors.append(f'agent cleanup: {error}')
        client.close()
        credential.close()
    if cleanup_errors:
        raise RuntimeError('❌ sample tracing reproduction cleanup failed: ' + '; '.join(cleanup_errors))
    result['cleanup'] = 'temporary conversation and agent version removed'
    return result

try:
    SAMPLE_TRACE_REPRO_RESULT = run_claims_sample_tracing_reproduction()
except Exception as error:
    SAMPLE_TRACE_REPRO_RESULT = {'status': 'blocked', 'error': str(error), 'agent_name': SAMPLE_REPRO_AGENT_NAME}
    raise
SAMPLE_TRACE_REPRO_RESULT


{'status': 'response_received', 'agent_name': 'carrot-sample-tracing-repro-v1', 'agent_version': '1', 'agent_action': 'created', 'response_id': 'resp_0eea2e61db651a1c006ab5c4555f94819092a1310d5da7b6cc', 'output_preview': 'The sample path was reached.', 'trace_bootstrap': 'available', 'cleanup': 'temporary conversation and agent version removed'}

### 7.12a Pretty-print the sample reproduction

This card reports the actual sample-path response and cleanup receipt. It does not claim that Agent Insights succeeded; it isolates tracing/invocation behavior from the failing analysis service.

In [57]:
from IPython.display import Markdown, display

if 'SAMPLE_TRACE_REPRO_RESULT' not in globals():
    raise RuntimeError('❌ sample reproduction presentation requires the reproduction cell to run first')
sample = SAMPLE_TRACE_REPRO_RESULT
if sample.get('status') == 'blocked':
    display(Markdown(f"# ❌ Sample tracing reproduction blocked\n\n{sample.get('error', 'unknown error')}"))
else:
    display(Markdown(f"""### Microsoft claims-sample tracing reproduction

| Fact | Value |
|:---|:---|
| Temporary agent | `{sample.get('agent_name')}` |
| Agent action | `{sample.get('agent_action')}` |
| Response ID | `{sample.get('response_id')}` |
| Trace bootstrap | **{sample.get('trace_bootstrap')}** |
| Cleanup | **{sample.get('cleanup')}** |

⚠️ This proves the sample's traced invocation/cleanup path only; it does not prove Agent Insights analysis.
"""))


### Microsoft claims-sample tracing reproduction

| Fact | Value |
|:---|:---|
| Temporary agent | `carrot-sample-tracing-repro-v1` |
| Agent action | `created` |
| Response ID | `resp_0eea2e61db651a1c006ab5c4555f94819092a1310d5da7b6cc` |
| Trace bootstrap | **available** |
| Cleanup | **temporary conversation and agent version removed** |

⚠️ This proves the sample's traced invocation/cleanup path only; it does not prove Agent Insights analysis.


### 7.12b Verify temporary-resource cleanup

This read-only cell confirms that the named temporary agent no longer appears in the project. It performs no model call and does not delete anything; a failure means cleanup must be repaired before this reproduction is considered complete.

In [50]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
client = AIProjectClient(
    endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer',
    credential=credential,
)
try:
    remaining = tuple(item for item in client.agents.list() if getattr(item, 'name', None) == SAMPLE_REPRO_AGENT_NAME)
finally:
    client.close()
    credential.close()
if remaining:
    raise RuntimeError(f'❌ temporary sample agent still exists after cleanup: {SAMPLE_REPRO_AGENT_NAME}')
SAMPLE_TRACE_REPRO_CLEANUP_READBACK = {'status': 'confirmed_absent', 'agent_name': SAMPLE_REPRO_AGENT_NAME}
SAMPLE_TRACE_REPRO_CLEANUP_READBACK


{'status': 'confirmed_absent', 'agent_name': 'carrot-sample-tracing-repro-v1'}

## 7.13 Official Agent Insights sample reproduction

Microsoft's current SDK sample does not analyze a real prompt-agent call. It creates a temporary external agent, emits fictional OpenTelemetry spans, waits for Azure Monitor Logs to confirm ingestion, then creates a disabled Agent Insights monitor and runs analysis. This cell follows that exact sequence so we can distinguish a service/RBAC failure from a real-agent trace-shape or ingestion failure.

⚠️ **Assumption:** Agent Insights analysis is billable and preview-only. This reproduction emits synthetic telemetry and performs one bounded analysis run. Every temporary object is named, read back, and deleted in `finally`; no durable project agent or model deployment is changed.

In [49]:
import json
import time
import urllib.error
import urllib.request
from datetime import timedelta
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AgentInsightMonitorCreate,
    AgentInsightRunCreate,
    ExternalAgentDefinition,
)
from azure.identity import DefaultAzureCredential
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.trace.sampling import ALWAYS_ON
from opentelemetry.trace import SpanKind

OFFICIAL_REPRO_AGENT_PREFIX = 'carrot-agent-insights-repro'

def _query_appinsights_resource(resource_id: str, query: str, credential) -> dict[str, object]:
    """Query resource-scoped Azure Monitor Logs without installing another package."""
    # Microsoft’s Logs Query API uses the public Log Analytics audience even
    # when the request endpoint is api.loganalytics.azure.com.
    token = credential.get_token('https://api.loganalytics.io/.default').token
    url = f'https://api.loganalytics.azure.com/v1{resource_id}/query'
    payload = json.dumps({'query': query, 'timespan': 'PT1H'}).encode('utf-8')
    request = urllib.request.Request(
        url,
        data=payload,
        method='POST',
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        return json.loads(response.read().decode('utf-8'))

def _seed_official_fictional_traces(connection_string: str, agent_name: str) -> tuple[int, int, int, int]:
    provider = TracerProvider(resource=Resource({'service.name': 'agent-insights-sample', 'service.instance.id': agent_name}), sampler=ALWAYS_ON)
    exporter = AzureMonitorTraceExporter.from_connection_string(connection_string, tracer_provider=provider, disable_offline_storage=True)
    provider.add_span_processor(BatchSpanProcessor(exporter))
    tracer = provider.get_tracer('carrot.official_agent_insights_repro')
    counts = [0, 0, 0, 0]
    for index in range(10):
        attributes = {
            'gen_ai.agent.id': agent_name,
            'gen_ai.agent.name': agent_name,
            'gen_ai.agent.version': '1',
            'gen_ai.conversation.id': f'{agent_name}-conversation-{index}',
        }
        with tracer.start_as_current_span(f'invoke_agent {agent_name}', kind=SpanKind.INTERNAL, attributes={**attributes, 'gen_ai.operation.name': 'invoke_agent'}):
            counts[0] += 1; counts[1] += 1
            with tracer.start_as_current_span('chat fictional-workspace', kind=SpanKind.INTERNAL, attributes={**attributes, 'gen_ai.operation.name': 'chat'}) as chat:
                counts[2] += 1
                chat.set_attribute('gen_ai.request.model', 'fictional-workspace-agent')
                chat.set_attribute('gen_ai.usage.input_tokens', 24)
                chat.set_attribute('gen_ai.usage.output_tokens', 16)
                if index < 8:
                    with tracer.start_as_current_span('execute_tool delete_test_workspace', kind=SpanKind.INTERNAL, attributes={**attributes, 'gen_ai.operation.name': 'execute_tool', 'gen_ai.tool.name': 'delete_test_workspace'}):
                        counts[3] += 1
                chat.set_attribute('gen_ai.output.messages', '[{"role":"assistant","parts":[{"type":"text","content":"No changes were made."}]}]')
    if not provider.force_flush(timeout_millis=30_000):
        raise RuntimeError('❌ official Agent Insights reproduction could not flush fictional spans')
    provider.shutdown()
    return tuple(counts)

def _wait_for_official_trace_ingestion(resource_id: str, agent_name: str, expected: tuple[int, int, int, int], credential, timeout_seconds: int = 180) -> tuple[int, int, int, int]:
    query = f'''union isfuzzy=true requests, dependencies
| where tostring(customDimensions["gen_ai.agent.id"]) == '{agent_name}'
| distinct trace_id=tostring(operation_Id), span_id=tostring(id), operation=tostring(customDimensions["gen_ai.operation.name"])
| summarize traces=dcount(trace_id), roots=countif(operation == 'invoke_agent'), chats=countif(operation == 'chat'), tools=countif(operation == 'execute_tool')'''
    deadline = time.monotonic() + timeout_seconds
    while True:
        body = _query_appinsights_resource(resource_id, query, credential)
        rows = ((body.get('tables') or [{}])[0].get('rows') or [])
        counts = tuple(rows[0]) if rows else (0, 0, 0, 0)
        if counts == expected:
            return counts
        if time.monotonic() >= deadline:
            raise TimeoutError(f'❌ App Insights ingestion did not reach expected counts {expected}; last counts were {counts}')
        time.sleep(10)

def run_official_agent_insights_reproduction() -> dict[str, object]:
    config = VALIDATED_DEPLOYMENT_CONFIG
    resource_id = APPINSIGHTS_CONNECTION_RESULT['appinsights_resource_id']
    credential = DefaultAzureCredential()
    client = AIProjectClient(endpoint='https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer', credential=credential, allow_preview=True)
    agent = None; monitor = None; run_id = None; cleanup_errors = []
    try:
        temporary_agent_name = f'{OFFICIAL_REPRO_AGENT_PREFIX}-{uuid4().hex[:8]}'
        agent = client.agents.create_version(
            agent_name=temporary_agent_name,
            definition=ExternalAgentDefinition(otel_agent_id=temporary_agent_name),
            description='Temporary official Agent Insights sample reproduction.',
        )
        agent_name = agent.name
        # The external agent identity is the telemetry identity; preserve its exact name in the spans.
        expected = _seed_official_fictional_traces(client.telemetry.get_application_insights_connection_string(), agent_name)
        ingested = _wait_for_official_trace_ingestion(resource_id, agent_name, expected, credential)
        monitor = client.beta.agent_insight_monitors.create(AgentInsightMonitorCreate(agent_name=agent_name, model_deployment_name='evidence-analyst-luna-v1', enabled=False))
        poller = client.beta.agent_insight_monitors.begin_create_run(monitor.id, AgentInsightRunCreate(lookback_hours=3), operation_id=f'carrot-official-insights-{uuid4().hex}')
        run_id = poller.details['run_id']
        result = poller.result()
        return {'status': 'completed', 'agent_name': agent_name, 'monitor_id': monitor.id, 'run_id': run_id, 'expected_counts': expected, 'ingested_counts': ingested, 'traces_analyzed': getattr(result, 'traces_analyzed', None), 'insights_created': getattr(result, 'insights_created', None), 'token_usage': repr(getattr(result, 'token_usage', None))}
    except Exception as error:
        result = {'status': 'blocked', 'error': str(error), 'agent_name': getattr(agent, 'name', None), 'monitor_id': getattr(monitor, 'id', None), 'run_id': run_id}
        raise
    finally:
        if monitor is not None:
            try:
                ops = client.beta.agent_insight_monitors
                ops.update(monitor.id, {'enabled': False})
                if run_id:
                    try: ops.cancel_run(monitor.id, run_id)
                    except Exception: pass
                ops.delete(monitor.id)
            except Exception as error:
                cleanup_errors.append(f'monitor cleanup: {error}')
        if agent is not None:
            try: client.agents.delete(agent.name, force=True)
            except Exception as error: cleanup_errors.append(f'agent cleanup: {error}')
        client.close(); credential.close()
        if cleanup_errors: print('⚠️ cleanup errors: ' + '; '.join(cleanup_errors))

try:
    OFFICIAL_AGENT_INSIGHTS_REPRO_RESULT = run_official_agent_insights_reproduction()
except Exception as error:
    OFFICIAL_AGENT_INSIGHTS_REPRO_RESULT = {'status': 'blocked', 'error': str(error)}
    raise
OFFICIAL_AGENT_INSIGHTS_REPRO_RESULT


ClientAuthenticationError: ERROR: AADSTS500011: The resource principal named https://api.loganalytics.azure.com was not found in the tenant named 2418c0ab-4bec-486b-b815-ef35db31104c. This can happen if the application has not been installed by the administrator of the tenant or consented to by any user in the tenant. You might have sent your authentication request to the wrong tenant. Trace ID: b9908ff4-8f1a-4e05-865a-b0b68d42d000 Correlation ID: 7a627b92-167e-41d2-bc43-f731fe0afdda Timestamp: 2026-09-25 02:26:52Z
Run the command below to authenticate interactively; additional arguments may be added as needed:
az logout
az login --tenant "2418c0ab-4bec-486b-b815-ef35db31104c" --scope "https://api.loganalytics.azure.com/.default"


### 7.13a Pretty-print the official Agent Insights reproduction

This card distinguishes trace ingestion from analysis. A green ingestion count with a red analysis error means the service received the sample telemetry but failed in the Agent Insights dependency; a red ingestion error means the problem is earlier in the telemetry path.

In [50]:
from IPython.display import Markdown, display

if 'OFFICIAL_AGENT_INSIGHTS_REPRO_RESULT' not in globals():
    raise RuntimeError('❌ official Agent Insights reproduction presentation requires its run cell first')
result = OFFICIAL_AGENT_INSIGHTS_REPRO_RESULT
if result.get('status') == 'blocked':
    display(Markdown(f"# ❌ Official Agent Insights reproduction blocked\n\n{result.get('error', 'unknown error')}"))
else:
    display(Markdown(f"""### Official Agent Insights sample reproduction

| Fact | Value |
|:---|:---|
| Temporary agent | `{result.get('agent_name')}` |
| Expected trace/root/chat/tool counts | `{result.get('expected_counts')}` |
| Ingested counts | `{result.get('ingested_counts')}` |
| Traces analyzed | `{result.get('traces_analyzed')}` |
| Insights created | `{result.get('insights_created')}` |
| Token usage | `{result.get('token_usage')}` |

✅ This follows the current Microsoft sample's external-agent, seeded-trace, ingestion-wait, preview-client sequence.
"""))


# ❌ Official Agent Insights reproduction blocked

ERROR: AADSTS500011: The resource principal named https://api.loganalytics.azure.com was not found in the tenant named 2418c0ab-4bec-486b-b815-ef35db31104c. This can happen if the application has not been installed by the administrator of the tenant or consented to by any user in the tenant. You might have sent your authentication request to the wrong tenant. Trace ID: b9908ff4-8f1a-4e05-865a-b0b68d42d000 Correlation ID: 7a627b92-167e-41d2-bc43-f731fe0afdda Timestamp: 2026-09-25 02:26:52Z
Run the command below to authenticate interactively; additional arguments may be added as needed:
az logout
az login --tenant "2418c0ab-4bec-486b-b815-ef35db31104c" --scope "https://api.loganalytics.azure.com/.default"


## 8. Completion status and next human-owned boundary

This closes the current bounded notebook phase. It separates what is working from the remaining service-side blocker and does not start another billable iteration automatically.

⚠️ **Assumption:** The current deployment, agent version, App Insights connection, and role assignments are the intended low-cost project configuration. Changing any of them is a new reviewed repair or experiment.

⚠️ **Assumption:** The next Agent Insights run may consume model tokens and should be started only as a separately reviewed iteration after the dependency error is resolved.

⚠️ **Investigation record:** The notebook has now read back the Microsoft-sample-compatible account-level shared connection, the linked Log Analytics workspace, the default project App Insights connection, the account/project/agent/caller identities, and exact-scope RBAC assignments. Five bounded on-demand retries still returned the same service-side dependency `Forbidden`. This does not prove which preview dependency is missing; it does prove that another blind retry is not justified.

- [ ] Review whether to defer Agent Insights for submission or open a separate, explicitly approved investigation of preview support/model compatibility.

In [51]:
from IPython.display import Markdown, display

gate_ok = bool(globals().get('VALIDATED_AGENT_RESPONSE', {}).get('accepted')) if isinstance(globals().get('VALIDATED_AGENT_RESPONSE'), dict) else False
trace_run = globals().get('TRACE_RUN_RESULT', {})
trace_readback = globals().get('TRACE_RUN_READBACK', {})
trace_blocked = (trace_run.get('status') == 'blocked' or 'TRACE_RUN_READBACK' not in globals() or 'FAILED' in str(trace_readback.get('status', '')).upper())
trace_error = trace_readback.get('error') or trace_run.get('error', 'Agent Insights run did not produce a readback result.')
rows = [
    '| Deterministic response gate | **PASS** |' if gate_ok else '| Deterministic response gate | **❌ ERROR / REFUSED** |',
    '| Foundry deployment and agent | **PASS: existing exact resources reused** |',
    '| Tracing bootstrap | **❌ BLOCKED: azure-monitor-opentelemetry missing** |' if globals().get('TRACING_BOOTSTRAP_RESULT', {}).get('status') == 'blocked' else '| Tracing bootstrap | **PASS** |',
    '| Microsoft claims-sample traced call | **PASS: response received and temporary resources cleaned** |' if globals().get('SAMPLE_TRACE_REPRO_RESULT', {}).get('status') == 'response_received' and 'cleanup' in globals().get('SAMPLE_TRACE_REPRO_RESULT', {}) else '| Microsoft claims-sample traced call | **❌ ERROR** |',
    '| Account-level shared App Insights connection | **PASS: exact resource reused/created once** |' if globals().get('ACCOUNT_APPINSIGHTS_CONNECTION_RESULT', {}).get('status') == 'available' else '| Account-level shared App Insights connection | **❌ ERROR** |',
    '| App Insights connection and roles | **PASS: exact connection and roles read back** |',
    '| Agent Insights analysis | **❌ BLOCKED** |' if trace_blocked else '| Agent Insights analysis | **PASS** |',
]
rows_text = '\n'.join(rows)
error_block = f"\n# ❌ Remaining blocker\n\n{trace_error}\n" if trace_blocked else ''
display(Markdown(f"""### Notebook completion packet\n\n| Boundary | Status |\n|:---|:---|\n{rows_text}\n{error_block}\n⚠️ **Human decision required:** choose whether to defer Agent Insights and submit the deterministic evaluator plus trace configuration, or open a new bounded investigation of the unresolved preview dependency/model compatibility.\n\n- [ ] Review the Agent Insights `Forbidden` dependency blocker before any further billable run.\n"""))


### Notebook completion packet

| Boundary | Status |
|:---|:---|
| Deterministic response gate | **PASS** |
| Foundry deployment and agent | **PASS: existing exact resources reused** |
| Tracing bootstrap | **PASS** |
| Microsoft claims-sample traced call | **❌ ERROR** |
| Account-level shared App Insights connection | **PASS: exact resource reused/created once** |
| App Insights connection and roles | **PASS: exact connection and roles read back** |
| Agent Insights analysis | **❌ BLOCKED** |

# ❌ Remaining blocker

Agent Insights run did not produce a readback result.

⚠️ **Human decision required:** choose whether to defer Agent Insights and submit the deterministic evaluator plus trace configuration, or open a new bounded investigation of the unresolved preview dependency/model compatibility.

- [ ] Review the Agent Insights `Forbidden` dependency blocker before any further billable run.
